# 08. Portfolio Construction

El objetivo de este notebook será transformar las predicciones OOS de Ridge, XGBoost y Random Forest en señales de inversión y posteriormente en carteras invertibles, evaluando cómo las distintas reglas de selección, asignación de pesos, restricciones y rebalanceo afectan al resultado económico.

El objetivo no es todavía realizar un análisis exhaustivo de rentabilidad y riesgo, sino construir de forma sistemática las carteras que posteriormente serán evaluadas en el Notebook 09.

## 1. Imports & Configuration

### 1.1 Librerías

In [1]:
import sys
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import random
import warnings
import joblib

from pathlib import Path
from scipy.stats import spearmanr

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

### 1.2 Configuración del notebook

El horizonte de predicción utilizado por los modelos es de 21 sesiones de trading, correspondiente al target forward_return_21d. Este horizonte define el periodo sobre el que se evalúa la capacidad predictiva de las señales, pero no determina automáticamente la frecuencia de rebalanceo de las carteras. Como configuración inicial se utilizará un rebalanceo cada 21 sesiones, que posteriormente se comparará con frecuencias alternativas para analizar su impacto sobre turnover, costes y estabilidad de las posiciones.

Las carteras benchmark se construirán a partir de los extremos del ranking cross-sectional, utilizando el 10% superior para posiciones long y, en estrategias long-short, el 10% inferior para posiciones short. Las exposiciones objetivo se mantendrán constantes entre modelos para garantizar una comparación homogénea, mientras que el número exacto de posiciones dependerá del universo disponible en cada fecha de rebalanceo.


In [2]:
# =============================================================================
# Experiment Configuration
# =============================================================================

# Prediction horizon
PREDICTION_HORIZON = 21

# Rebalancing
# Benchmark frequency; alternative frequencies will be evaluated later.
REBALANCING_FREQUENCY = 21

# Portfolio selection
SELECTION_PERCENTS = [0.10, 0.20, 0.30]

# Benchmark selection
BENCHMARK_SELECTION = 0.10

# Long-only exposure
LONG_ONLY_GROSS_EXPOSURE = 1.00

# Long-short exposure
LONG_SHORT_LONG_EXPOSURE = 0.50
LONG_SHORT_SHORT_EXPOSURE = 0.50

# Position selection
POSITION_COUNT_MODE = "quantile"

In [3]:
# =============================================================================
# Exposure Validation
# =============================================================================

assert (
    LONG_SHORT_LONG_EXPOSURE >= 0
    and LONG_SHORT_SHORT_EXPOSURE >= 0
)

assert (
    LONG_SHORT_LONG_EXPOSURE
    + LONG_SHORT_SHORT_EXPOSURE
    > 0
)

### 1.3 Rutas y parámetros globales

In [4]:
# =============================================================================
# Paths
# =============================================================================

OOS_DATA_PATH = "../data/model_results/final_model/oos/"
PREDICTIONS_PATH = (OOS_DATA_PATH + "oos_predictions.parquet")

# Path to price data (OOS included)
PRICES_PATH = "../data/raw/sp500_prices_extended.parquet"

# =============================================================================
# Models
# =============================================================================

MODEL_COLUMNS = {
    "Ridge": "prediction_ridge_rank",
    "XGBoost": "prediction_xgb_rank",
    "Random Forest": "prediction_rf_rank",
}

# =============================================================================
# Required Columns
# =============================================================================

REQUIRED_COLUMNS = [
    "prediction_ridge_rank",
    "prediction_xgb_rank",
    "prediction_rf_rank",
    "forward_return_21d",
]

# =============================================================================
# Signal Configuration
# =============================================================================

PREDICTION_COLUMNS = list(MODEL_COLUMNS.values())

RANKING_METHOD = "cross_sectional_percentile"

## 2. Data Loading

### 2.1 Carga de predicciones OOS

In [5]:
# =============================================================================
# Load OOS Predictions
# =============================================================================

oos_predictions = pd.read_parquet(PREDICTIONS_PATH)

print(f"OOS predictions shape: {oos_predictions.shape}")
print(f"Date range: {oos_predictions.index.get_level_values('date').min()} "
      f"→ {oos_predictions.index.get_level_values('date').max()}")

OOS predictions shape: (184408, 7)
Date range: 2025-01-15 00:00:00 → 2026-07-10 00:00:00


In [6]:
# =============================================================================
# Prediction Columns Check
# =============================================================================

missing_prediction_columns = [
    column
    for column in PREDICTION_COLUMNS
    if column not in oos_predictions.columns
]

assert not missing_prediction_columns, (
    f"Missing prediction columns: {missing_prediction_columns}"
)

# =============================================================================
# Index Check
# =============================================================================

assert isinstance(oos_predictions.index, pd.MultiIndex)
assert oos_predictions.index.names == ["date", "ticker"]

# =============================================================================
# Prediction Availability Check
# =============================================================================

assert oos_predictions[PREDICTION_COLUMNS].notna().any().all(), (
    "At least one prediction column contains no valid observations."
)

### 2.2 Carga de precios y retornos realizados

Se cargan los precios históricos necesarios para el backtest y se restringe el universo al conjunto de activos utilizado por las predicciones OOS. Se conserva toda la cobertura histórica disponible, ya que los precios anteriores al periodo OOS pueden ser necesarios para estimar medidas de riesgo en cada fecha de rebalanceo. A partir de los precios ajustados se calculan los retornos simples y logarítmicos, que se utilizarán posteriormente en la construcción y evaluación de las carteras.

In [7]:
# =============================================================================
# Load Extended Price Data
# =============================================================================

prices = pd.read_parquet(PRICES_PATH)

# =============================================================================
# Remove Securities Excluded During Development
# =============================================================================

tickers_to_remove = [
    "SW",
    "AMCR",
]

prices = prices.drop(
    columns=tickers_to_remove,
    level=1,
    errors="ignore",
)

# =============================================================================
# Align Price Universe with OOS Universe
# =============================================================================

oos_tickers = oos_predictions.index.get_level_values("ticker").unique()

price_tickers = prices.columns.get_level_values(1)

missing_price_tickers = oos_tickers.difference(price_tickers)

assert len(missing_price_tickers) == 0, (
    f"Missing price data for tickers: "
    f"{missing_price_tickers.tolist()}"
)

prices = prices.loc[
    :,
    prices.columns.get_level_values(1).isin(oos_tickers)
]

# =============================================================================
# Compute Returns
# =============================================================================

adj_close = prices["Adj Close"]

simple_returns = adj_close.pct_change()

log_returns = np.log(
    adj_close / adj_close.shift(1)
)

In [8]:
print(
    f"Price data coverage: "
    f"{prices.index.min().date()} → {prices.index.max().date()}"
)

print(f"Number of securities: {len(oos_tickers)}")

Price data coverage: 2010-01-04 → 2026-08-10
Number of securities: 496


### 2.3 Verificación de integridad y cobertura temporal

Antes de comenzar la construcción de las señales y carteras, se verifica que las predicciones OOS, los precios y los retornos presentan una cobertura temporal y un universo de activos compatibles. Se comprueba que todos los activos y fechas necesarios para el periodo OOS disponen de información de precios y que las series de retornos permanecen correctamente alineadas con los precios originales. Estas comprobaciones garantizan la consistencia de los datos utilizados en el backtest, mientras que la prevención de look-ahead bias se controlará posteriormente en cada etapa en la que se utilice información histórica para determinar señales, riesgo o pesos de cartera.

In [9]:
# =============================================================================
# Temporal Coverage
# =============================================================================

oos_dates = (
    oos_predictions.index
    .get_level_values("date")
    .unique()
)

assert oos_dates.min() >= prices.index.min()
assert oos_dates.max() <= prices.index.max()

# =============================================================================
# Ticker Coverage
# =============================================================================

oos_tickers = (
    oos_predictions.index
    .get_level_values("ticker")
    .unique()
)

price_tickers = (
    prices.columns
    .get_level_values(1)
    .unique()
)

missing_price_tickers = oos_tickers.difference(price_tickers)

assert len(missing_price_tickers) == 0, (
    f"Missing price data for tickers: "
    f"{missing_price_tickers.tolist()}"
)

# =============================================================================
# Prediction Date Coverage
# =============================================================================

missing_prediction_dates = oos_dates.difference(
    prices.index
)

assert len(missing_prediction_dates) == 0, (
    f"Missing price data for OOS dates: "
    f"{missing_prediction_dates.tolist()}"
)

# =============================================================================
# Returns Alignment
# =============================================================================

assert simple_returns.index.equals(prices.index)
assert log_returns.index.equals(prices.index)

assert simple_returns.columns.equals(adj_close.columns)
assert log_returns.columns.equals(adj_close.columns)

# =============================================================================
# Data Availability
# =============================================================================

assert simple_returns.notna().any().all(), (
    "Some securities contain no valid simple returns."
)

assert log_returns.notna().any().all(), (
    "Some securities contain no valid log returns."
)

## 3. Signal Construction

### 3.1 Cross-sectional Ranking

Las predicciones OOS de Ridge, XGBoost y Random Forest se transforman en rankings cross-sectional diarios para ordenar los activos según la intensidad relativa de su señal. El ranking se calcula únicamente sobre los activos que disponen de una predicción válida en cada fecha, respetando el universo efectivamente disponible durante el periodo OOS. Dado que los modelos basados en árboles presentan un número reducido de valores de predicción únicos, especialmente XGBoost, se utiliza un ranking determinista (method="first") para resolver los empates y garantizar una distribución aproximadamente equilibrada de los activos en los posteriores grupos de selección. Esta transformación no modifica las predicciones OOS originales.

In [10]:
# =============================================================================
# Cross-sectional Ranking
# =============================================================================

signal_ranks = pd.DataFrame(
    index=oos_predictions.index
)

RANK_COLUMNS = {}

for model, prediction_column in MODEL_COLUMNS.items():

    rank_column = f"{model.lower().replace(' ', '_')}_rank"

    signal_ranks[rank_column] = (
        oos_predictions
        .groupby(level="date")[prediction_column]
        .rank(
            method="first",
            pct=True,
        )
    )

    RANK_COLUMNS[model] = rank_column

In [11]:
# =============================================================================
# Ranking Validation
# =============================================================================

for rank_column in RANK_COLUMNS.values():

    assert signal_ranks[rank_column].dropna().between(0, 1).all(), (
        f"Invalid values found in {rank_column}."
    )

# =============================================================================
# Cross-sectional Coverage
# =============================================================================

rank_counts = (
    signal_ranks
    .groupby(level="date")
    .size()
)

print(
    f"Cross-sectional observations per date: "
    f"Min = {rank_counts.min()} | "
    f"Max = {rank_counts.max()}"
)

Cross-sectional observations per date: Min = 494 | Max = 496


### 3.2 Quantile Formation

El ranking cross-sectional continuo se divide en diez grupos de tamaño aproximadamente equivalente, desde D1, que contiene los activos con las predicciones relativas más bajas, hasta D10, que contiene los activos con las predicciones más altas. La formación de los grupos se realiza diariamente, utilizando únicamente los activos disponibles en cada fecha.

Debido al elevado número de empates presente en las predicciones de los modelos basados en árboles, especialmente XGBoost, se utiliza un ranking determinista (method="first") para resolver los empates antes de formar los cuantiles. De esta forma se evitan grupos excesivamente desiguales y se garantiza una comparación homogénea entre modelos. Esta decisión únicamente afecta a la asignación de observaciones a grupos y no modifica las predicciones originales.


In [12]:
# =============================================================================
# Quantile Formation
# =============================================================================

quantile_data = pd.DataFrame(
    index=signal_ranks.index
)

QUANTILE_COLUMNS = {}

for model, rank_column in RANK_COLUMNS.items():

    quantile_column = (
        f"{model.lower().replace(' ', '_')}_quantile"
    )

    quantile_data[quantile_column] = (
        np.ceil(
            signal_ranks[rank_column] * 10
        )
        .clip(upper=10)
        .astype("Int64")
    )

    QUANTILE_COLUMNS[model] = quantile_column

In [13]:
# =============================================================================
# Quantile Distribution
# =============================================================================

quantile_counts = {}

for model, quantile_column in QUANTILE_COLUMNS.items():

    counts = (
        quantile_data
        .groupby(level="date")[quantile_column]
        .value_counts()
        .unstack(fill_value=0)
    )

    quantile_counts[model] = counts

for model, counts in quantile_counts.items():

    print("=" * 80)
    print(f"{model} — QUANTILE SIZE RANGE")
    print("=" * 80)

    print(
        f"Min = {counts.min().min()} | "
        f"Max = {counts.max().max()}"
    )

Ridge — QUANTILE SIZE RANGE
Min = 49 | Max = 50
XGBoost — QUANTILE SIZE RANGE
Min = 49 | Max = 50
Random Forest — QUANTILE SIZE RANGE
Min = 49 | Max = 50


### 3.3 Signal Distribution Analysis

Antes de utilizar los rankings para construir las carteras, se analiza su distribución cross-sectional durante el periodo OOS. El objetivo es comprobar que las señales presentan suficiente dispersión entre activos y detectar posibles concentraciones, valores extremos o diferencias estructurales entre modelos que puedan afectar a la selección y posterior asignación de pesos. Dado que los rankings se construyen mediante percentiles cross-sectional, se presta especial atención a la distribución de las predicciones originales y al comportamiento de los grupos por cuantiles.


#### 3.3.1 Distribución de las predicciones

In [14]:
# =============================================================================
# Prediction Distribution
# =============================================================================

prediction_distribution = (
    oos_predictions[
        list(MODEL_COLUMNS.values())
    ]
    .describe()
    .T[
        [
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "50%",
            "75%",
            "max",
        ]
    ]
)

display(prediction_distribution)

# =============================================================================
# Unique Prediction Values
# =============================================================================

unique_predictions = pd.DataFrame(
    {
        model: [
            oos_predictions[prediction_column]
            .nunique()
        ]
        for model, prediction_column in MODEL_COLUMNS.items()
    },
    index=["Unique values"],
).T

display(unique_predictions)

,count,mean,std,min,25%,50%,75%,max
prediction_ridge_rank,184408.0,0.014807,0.004190,0.005012,0.011615,0.014853,0.017929,0.024854
prediction_xgb_rank,184408.0,0.014821,0.003660,0.013136,0.013136,0.013318,0.015174,0.057154
prediction_rf_rank,184408.0,0.015472,0.009112,0.011264,0.012054,0.013353,0.015931,0.159783


,Unique values
Ridge,182612
XGBoost,285
Random Forest,4680


El análisis descriptivo de las predicciones fuera de muestra revela marcadas diferencias en la naturaleza estadística de cada arquitectura. Mientras que el modelo lineal Ridge genera una distribución continua con una elevada granulatividad (182,612 valores únicos), los modelos basados en árboles muestran una clara concentración discreta. 

En particular, XGBoost comprime el universo de predicciones en apenas 285 valores únicos, provocando que más del $50\%$ de las observaciones colapsen en un valor idéntico ($0.013136$).Por su parte, Random Forest presenta una mayor variabilidad ($\sigma = 0.009112$) y una pronunciada asimetría hacia la cola derecha, alcanzando predicciones puntuales de hasta el $15.97\%$. 

A pesar de estas diferencias morfológicas en la distribución de las salidas, los tres modelos mantienen una media de retorno predicho altamente coherente entre sí ($\approx 1.48\% - 1.54\%$). Esta acusada presencia de empates en XGBoost y Random Forest ratifica la necesidad de aplicar un ranking determinista (method="first") para garantizar una división equitativa y homogénea del universo en cuantiles.

#### 3.3.2 Distribución de los rankings

In [15]:
# =============================================================================
# Ranking Distribution
# =============================================================================

rank_distribution = (
    signal_ranks
    .describe()
    .T[
        [
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "50%",
            "75%",
            "max",
        ]
    ]
)

display(rank_distribution)

,count,mean,std,min,25%,50%,75%,max
ridge_rank,184408.0,0.501009,0.288675,0.002016,0.251012,0.50101,0.751012,1.0
xgboost_rank,184408.0,0.501009,0.288675,0.002016,0.251012,0.50101,0.751012,1.0
random_forest_rank,184408.0,0.501009,0.288675,0.002016,0.251012,0.50101,0.751012,1.0


Tras la transformación a percentiles diarios ($[0, 1]$), la distribución de las tres señales converge exactamente a una distribución uniforme teórica $\mathcal{U}(0, 1)$.Los tres modelos presentan una media y mediana centradas en $0.5010$ y una desviación estándar idéntica a la teórica ($\sigma \approx 0.2887$). 

Esta estandarización elimina las diferencias de escala previas entre algoritmos, garantizando un punto de partida homogéneo y simétrico para la división en deciles ($D1$ a $D10$).

#### 3.3.3 Distribución de los cuantiles

In [16]:
# =============================================================================
# Quantile Distribution
# =============================================================================

quantile_distribution = {}

for model, quantile_column in QUANTILE_COLUMNS.items():

    counts = (
        quantile_data[quantile_column]
        .value_counts()
        .sort_index()
    )

    quantile_distribution[model] = counts

quantile_distribution = pd.DataFrame(
    quantile_distribution
)

quantile_distribution.index.name = "Quantile"

display(quantile_distribution)

# =============================================================================
# Quantile Imbalance
# =============================================================================

quantile_imbalance = {}

for model, quantile_column in QUANTILE_COLUMNS.items():

    counts = (
        quantile_data
        .groupby(level="date")[quantile_column]
        .value_counts()
        .unstack(fill_value=0)
    )

    quantile_imbalance[model] = {
        "Minimum group size": counts.min().min(),
        "Maximum group size": counts.max().max(),
        "Maximum imbalance": (
            counts.max().max()
            - counts.min().min()
        ),
    }

quantile_imbalance = pd.DataFrame(
    quantile_imbalance
).T

display(quantile_imbalance)

,Ridge,XGBoost,Random Forest
Quantile,,,
1,18228,18228,18228
2,18550,18550,18550
3,18278,18278,18278
4,18550,18550,18550
5,18596,18596,18596
6,18232,18232,18232
7,18546,18546,18546
8,18282,18282,18282
9,18546,18546,18546


,Minimum group size,Maximum group size,Maximum imbalance
Ridge,49,50,1
XGBoost,49,50,1
Random Forest,49,50,1



## 4. Benchmark Portfolio Formation

Antes de introducir metodologías de asignación de pesos basadas en la intensidad de la señal, el riesgo o la optimización, se construyen carteras benchmark mediante reglas simples y transparentes. Estas carteras permiten establecer una referencia común frente a la que evaluar posteriormente si los métodos de weighting más sofisticados aportan mejoras en términos de rentabilidad, riesgo, concentración o costes de implementación.

Se utilizarán dos benchmarks principales. El primero será una estrategia long-only basada en el decil superior del ranking, que servirá como referencia para evaluar diferentes metodologías de asignación dentro de una cartera con exposición positiva. El segundo será una estrategia long-short que combina el decil superior e inferior, permitiendo evaluar la capacidad de las señales para diferenciar entre activos con expectativas relativas altas y bajas.

En ambos casos se utilizará Equal Weight como regla de asignación inicial. De esta forma, la selección de activos y la asignación de pesos quedan separadas: los rankings determinan qué activos entran en la cartera, mientras que el benchmark asigna el mismo peso a todas las posiciones seleccionadas. Estas carteras constituirán la referencia frente a la que se compararán posteriormente las metodologías de Signal Weighting, Risk-Based Allocation y Mathematical Optimization.


In [17]:
# =============================================================================
# Portfolio Weights Structure
# =============================================================================

portfolio_weights = pd.DataFrame(
    columns=[
        "date",
        "ticker",
        "model",
        "portfolio",
        "weight",
    ]
)


### 4.1 Long-only Top 10% Equal Weight

Se construye una cartera long-only utilizando el decil superior del ranking de cada modelo como universo de inversión. Todos los activos seleccionados reciben el mismo peso, de forma que la cartera mantiene una exposición bruta y neta de 100% sin introducir información adicional sobre la intensidad de la señal o el riesgo individual de los activos. Esta estrategia constituye el benchmark principal para evaluar posteriormente si metodologías de weighting más sofisticadas aportan valor adicional respecto a una asignación simple y robusta. La selección se realiza de forma independiente para cada modelo y fecha de rebalanceo, y los pesos se almacenan en una estructura común para facilitar su posterior comparación.


In [18]:
# =============================================================================
# Long-only Top 10% Equal Weight
# =============================================================================

long_only_weights_list = []

for model, quantile_column in QUANTILE_COLUMNS.items():

    # -------------------------------------------------------------------------
    # Select top decile
    # -------------------------------------------------------------------------

    selected = (
        quantile_data[quantile_column] == 10
    )

    selected_index = (
        quantile_data.index[selected]
    )

    # -------------------------------------------------------------------------
    # Count selected positions per date
    # -------------------------------------------------------------------------

    selected_counts = (
        pd.Series(
            1,
            index=selected_index,
        )
        .groupby(level="date")
        .sum()
    )

    # -------------------------------------------------------------------------
    # Equal weights
    # -------------------------------------------------------------------------

    weights = (
        pd.Series(
            1.0,
            index=selected_index,
        )
        .div(
            selected_counts,
            level="date",
        )
    )

    # -------------------------------------------------------------------------
    # Build model portfolio
    # -------------------------------------------------------------------------

    model_weights = (
        weights
        .rename("weight")
        .reset_index()
    )

    model_weights["model"] = model
    model_weights["portfolio"] = (
        "long_only_equal_weight"
    )

    long_only_weights_list.append(
        model_weights[
            [
                "date",
                "ticker",
                "model",
                "portfolio",
                "weight",
            ]
        ]
    )


# =============================================================================
# Combine models
# =============================================================================

long_only_weights = pd.concat(
    long_only_weights_list,
    ignore_index=True,
)

In [19]:
# =============================================================================
# Validation
# =============================================================================

print("=" * 80)
print("LONG-ONLY TOP 10% EQUAL WEIGHT — VALIDATION")
print("=" * 80)


# =============================================================================
# 1. No negative weights
# =============================================================================

assert (
    long_only_weights["weight"] >= 0
).all(), (
    "Negative weights detected."
)

print("✓ All portfolio weights are non-negative.")


# =============================================================================
# 2. One observation per ticker, date and model
# =============================================================================

duplicates = (
    long_only_weights
    .duplicated(
        subset=[
            "date",
            "ticker",
            "model",
        ]
    )
    .sum()
)

assert duplicates == 0, (
    f"Found {duplicates} duplicated "
    "date-ticker-model observations."
)

print(
    "✓ No duplicated date-ticker-model observations."
)


# =============================================================================
# 3. Portfolio exposure
# =============================================================================

long_only_exposure = (
    long_only_weights
    .groupby(
        [
            "date",
            "model",
        ]
    )["weight"]
    .sum()
)

assert np.allclose(
    long_only_exposure.values,
    LONG_ONLY_GROSS_EXPOSURE,
), (
    "Portfolio exposure is not equal to "
    f"{LONG_ONLY_GROSS_EXPOSURE:.2f}."
)

print(
    f"✓ Portfolio exposure = "
    f"{LONG_ONLY_GROSS_EXPOSURE:.2f}."
)


# =============================================================================
# 4. Equal-weight validation
# =============================================================================

weight_check = (
    long_only_weights
    .groupby(
        [
            "date",
            "model",
        ]
    )["weight"]
    .agg(
        min_weight="min",
        max_weight="max",
    )
)

assert np.allclose(
    weight_check["min_weight"],
    weight_check["max_weight"],
), (
    "Weights are not equal within at least "
    "one portfolio."
)

print("✓ Equal weighting confirmed.")


# =============================================================================
# 5. Position count
# =============================================================================

position_count = (
    long_only_weights
    .groupby(
        [
            "date",
            "model",
        ]
    )["ticker"]
    .nunique()
)

print(
    "✓ Position count:"
)

print(
    f"  Min = {position_count.min()}"
)

print(
    f"  Max = {position_count.max()}"
)


# =============================================================================
# 6. Check expected Top 10% size
# =============================================================================

expected_position_count = (
    quantile_data
    .groupby(level="date")
    .size()
    * BENCHMARK_SELECTION
)

expected_position_count = (
    expected_position_count
    .round()
    .astype(int)
)

actual_position_count = (
    position_count
    .groupby(level="date")
    .first()
)

print(
    "✓ Expected Top 10% position count:"
)

print(
    f"  Min = {expected_position_count.min()}"
)

print(
    f"  Max = {expected_position_count.max()}"
)


# =============================================================================
# 7. Weight range
# =============================================================================

print(
    "✓ Weight range:"
)

print(
    f"  Min = {long_only_weights['weight'].min():.6f}"
)

print(
    f"  Max = {long_only_weights['weight'].max():.6f}"
)


# =============================================================================
# 8. Final summary
# =============================================================================

print("\n" + "=" * 80)
print("VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 80)

LONG-ONLY TOP 10% EQUAL WEIGHT — VALIDATION
✓ All portfolio weights are non-negative.
✓ No duplicated date-ticker-model observations.
✓ Portfolio exposure = 1.00.
✓ Equal weighting confirmed.
✓ Position count:
  Min = 50
  Max = 50
✓ Expected Top 10% position count:
  Min = 49
  Max = 50
✓ Weight range:
  Min = 0.020000
  Max = 0.020000

VALIDATION COMPLETED SUCCESSFULLY


### 4.2 Long-short D10/D1 Equal Weight

Se construye una cartera long-short combinando el decil superior (D10) y el decil inferior (D1) del ranking de cada modelo. Los activos de D10 reciben posiciones largas con una exposición agregada del 50%, mientras que los activos de D1 reciben posiciones cortas con una exposición agregada del 50%. Dentro de cada lado se utiliza Equal Weight, de forma que todos los activos seleccionados reciben la misma magnitud de peso dentro de su respectivo lado.

Esta configuración genera una cartera market-neutral en términos de exposición, con una exposición neta objetivo de 0% y una exposición bruta de 100%. De este modo, el rendimiento de la estrategia depende principalmente de la capacidad del modelo para diferenciar entre los activos situados en los extremos superior e inferior del ranking, en lugar de depender de una exposición direccional al mercado.

La elección de una exposición del 50% por lado se mantiene constante para los tres modelos y constituye el benchmark frente al que posteriormente se evaluarán otras metodologías de weighting.

In [20]:
# =============================================================================
# 4.2 Long-short D10/D1 Equal Weight
# =============================================================================

long_short_weights_list = []

TARGET_LONG_EXPOSURE = 0.50
TARGET_SHORT_EXPOSURE = -0.50

for model, quantile_column in QUANTILE_COLUMNS.items():

    # -------------------------------------------------------------------------
    # Select D10 (Longs) and D1 (Shorts)
    # -------------------------------------------------------------------------
    selected_long = quantile_data[quantile_column] == 10
    selected_short = quantile_data[quantile_column] == 1

    index_long = quantile_data.index[selected_long]
    index_short = quantile_data.index[selected_short]

    # -------------------------------------------------------------------------
    # Count selected positions per date
    # -------------------------------------------------------------------------
    count_long = (
        pd.Series(1, index=index_long)
        .groupby(level="date")
        .sum()
    )

    count_short = (
        pd.Series(1, index=index_short)
        .groupby(level="date")
        .sum()
    )

    # -------------------------------------------------------------------------
    # Calculate equal weights per side (+0.50 / N_long and -0.50 / N_short)
    # -------------------------------------------------------------------------
    weights_long = (
        pd.Series(TARGET_LONG_EXPOSURE, index=index_long)
        .div(count_long, level="date")
    )

    weights_short = (
        pd.Series(TARGET_SHORT_EXPOSURE, index=index_short)
        .div(count_short, level="date")
    )

    # -------------------------------------------------------------------------
    # Combine sides and build model portfolio
    # -------------------------------------------------------------------------
    combined_weights = pd.concat([weights_long, weights_short]).sort_index()

    model_weights = (
        combined_weights
        .rename("weight")
        .reset_index()
    )

    model_weights["model"] = model
    model_weights["portfolio"] = "long_short_equal_weight"

    long_short_weights_list.append(
        model_weights[
            [
                "date",
                "ticker",
                "model",
                "portfolio",
                "weight",
            ]
        ]
    )

# =============================================================================
# Combine models
# =============================================================================

long_short_weights = pd.concat(
    long_short_weights_list,
    ignore_index=True,
)

In [21]:
# =============================================================================
# Validation
# =============================================================================

print("=" * 80)
print("LONG-SHORT D10/D1 EQUAL WEIGHT — VALIDATION")
print("=" * 80)


# =============================================================================
# 1. No duplicated observations
# =============================================================================

duplicates = (
    long_short_weights
    .duplicated(
        subset=[
            "date",
            "ticker",
            "model",
        ]
    )
    .sum()
)

assert duplicates == 0, (
    f"Found {duplicates} duplicated date-ticker-model observations."
)

print("✓ No duplicated date-ticker-model observations.")


# =============================================================================
# 2. Side exposure verification (Long = +0.50, Short = -0.50)
# =============================================================================

long_exposure = (
    long_short_weights[long_short_weights["weight"] > 0]
    .groupby(["date", "model"])["weight"]
    .sum()
)

short_exposure = (
    long_short_weights[long_short_weights["weight"] < 0]
    .groupby(["date", "model"])["weight"]
    .sum()
)

assert np.allclose(long_exposure.values, 0.50), (
    "Long side exposure is not equal to +0.50."
)

assert np.allclose(short_exposure.values, -0.50), (
    "Short side exposure is not equal to -0.50."
)

print("✓ Side exposures confirmed: Long = +0.50 | Short = -0.50.")


# =============================================================================
# 3. Gross and Net exposure verification (Gross = 1.00, Net = 0.00)
# =============================================================================

gross_exposure = (
    long_short_weights
    .assign(abs_weight=long_short_weights["weight"].abs())
    .groupby(["date", "model"])["abs_weight"]
    .sum()
)

net_exposure = (
    long_short_weights
    .groupby(["date", "model"])["weight"]
    .sum()
)

assert np.allclose(gross_exposure.values, 1.00), (
    "Gross exposure is not equal to 1.00."
)

assert np.allclose(net_exposure.values, 0.00, atol=1e-8), (
    "Net exposure is not equal to 0.00."
)

print("✓ Portfolio exposure metrics confirmed: Gross = 1.00 | Net = 0.00.")


# =============================================================================
# 4. Equal weighting per side
# =============================================================================

long_check = (
    long_short_weights[long_short_weights["weight"] > 0]
    .groupby(["date", "model"])["weight"]
    .agg(min_w="min", max_w="max")
)

short_check = (
    long_short_weights[long_short_weights["weight"] < 0]
    .groupby(["date", "model"])["weight"]
    .agg(min_w="min", max_w="max")
)

assert np.allclose(long_check["min_w"], long_check["max_w"]), (
    "Weights on the Long side are not equal."
)

assert np.allclose(short_check["min_w"], short_check["max_w"]), (
    "Weights on the Short side are not equal."
)

print("✓ Equal weighting per side confirmed.")


# =============================================================================
# 5. Position count per side (Longs vs Shorts)
# =============================================================================

long_counts = (
    long_short_weights[long_short_weights["weight"] > 0]
    .groupby(["date", "model"])["ticker"]
    .nunique()
)

short_counts = (
    long_short_weights[long_short_weights["weight"] < 0]
    .groupby(["date", "model"])["ticker"]
    .nunique()
)

print("✓ Position counts:")
print(f"  Long side  (D10): Min = {long_counts.min()} | Max = {long_counts.max()}")
print(f"  Short side  (D1): Min = {short_counts.min()} | Max = {short_counts.max()}")


# =============================================================================
# 6. Weight magnitude range
# =============================================================================

min_long_weight = long_short_weights[long_short_weights["weight"] > 0]["weight"].min()
max_long_weight = long_short_weights[long_short_weights["weight"] > 0]["weight"].max()

min_short_weight = long_short_weights[long_short_weights["weight"] < 0]["weight"].min()
max_short_weight = long_short_weights[long_short_weights["weight"] < 0]["weight"].max()

print("✓ Weight magnitude ranges:")
print(f"  Long weights  : [{min_long_weight:.6f}, {max_long_weight:.6f}]")
print(f"  Short weights : [{min_short_weight:.6f}, {max_short_weight:.6f}]")


# =============================================================================
# 7. Final summary
# =============================================================================

print("\n" + "=" * 80)
print("LONG-SHORT VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 80)

LONG-SHORT D10/D1 EQUAL WEIGHT — VALIDATION
✓ No duplicated date-ticker-model observations.
✓ Side exposures confirmed: Long = +0.50 | Short = -0.50.
✓ Portfolio exposure metrics confirmed: Gross = 1.00 | Net = 0.00.
✓ Equal weighting per side confirmed.
✓ Position counts:
  Long side  (D10): Min = 50 | Max = 50
  Short side  (D1): Min = 49 | Max = 49
✓ Weight magnitude ranges:
  Long weights  : [0.010000, 0.010000]
  Short weights : [-0.010204, -0.010204]

LONG-SHORT VALIDATION COMPLETED SUCCESSFULLY



### 4.3 Portfolio Exposure

En este apartado se sintetizan y consolidan las métricas agregadas de exposición para las carteras de referencia construidas en las secciones anteriores (*Long-Only Top 10%* y *Long-Short D10/D1*). Para cualquier cartera $p$ en una fecha $t$, las exposiciones se definen de la siguiente forma:

- **Exposición Larga ($E_L$):** Suma de los pesos positivos, $E_{L,t} = \sum_{w_{i,t} > 0} w_{i,t}$
- **Exposición Corta ($E_S$):** Suma de los pesos negativos, $E_{S,t} = \sum_{w_{i,t} < 0} w_{i,t}$
- **Exposición Bruta ($E_{\text{gross}}$):** Suma de las magnitudes absolutas de los pesos, $E_{\text{gross},t} = \sum |w_{i,t}| = E_{L,t} + |E_{S,t}|$
- **Exposición Neta ($E_{\text{net}}$):** Suma algebraica de todos los pesos, $E_{\text{net},t} = \sum w_{i,t} = E_{L,t} + E_{S,t}$

A continuación se calcula el perfil de exposición diario medio para cada combinación de modelo y cartera con el fin de verificar formalmente el perfil de exposición de cada cartera y comprobar la neutralidad de exposición neta en la estrategia long-short, así como la invariancia de la exposición entre modelos.

In [22]:
benchmark_portfolios = pd.concat(
    [long_only_weights, long_short_weights],
    ignore_index=True,
)

daily_exposure = (
    benchmark_portfolios
    .groupby(["portfolio", "model", "date"])
    .agg(
        long_exp=("weight", lambda x: x[x > 0].sum()),
        short_exp=("weight", lambda x: x[x < 0].sum()),
        gross_exp=("weight", lambda x: x.abs().sum()),
        net_exp=("weight", "sum"),
    )
)

exposure_summary = (
    daily_exposure
    .groupby(["portfolio", "model"])
    .mean()
    .round(2)
    .reset_index()
)

print("=" * 80)
print("BENCHMARK PORTFOLIOS — EXPOSURE SUMMARY")
print("=" * 80)
display(exposure_summary.style.hide(axis="index"))

# =============================================================================
# Validation
# =============================================================================

# Silent assertions for Long-Only
lo_mask = exposure_summary["portfolio"] == "long_only_equal_weight"
assert np.allclose(exposure_summary.loc[lo_mask, "gross_exp"], 1.00), "Error in Long-Only Gross Exposure"
assert np.allclose(exposure_summary.loc[lo_mask, "net_exp"], 1.00), "Error in Long-Only Net Exposure"
assert np.allclose(exposure_summary.loc[lo_mask, "short_exp"], 0.00), "Error in Long-Only Short Exposure"

# Silent assertions for Long-Short
ls_mask = exposure_summary["portfolio"] == "long_short_equal_weight"
assert np.allclose(exposure_summary.loc[ls_mask, "gross_exp"], 1.00), "Error in Long-Short Gross Exposure"
assert np.allclose(exposure_summary.loc[ls_mask, "net_exp"], 0.00, atol=1e-6), "Error in Long-Short Net Exposure"
assert np.allclose(exposure_summary.loc[ls_mask, "long_exp"], 0.50), "Error in Long-Short Long Exposure"
assert np.allclose(exposure_summary.loc[ls_mask, "short_exp"], -0.50), "Error in Long-Short Short Exposure"

print("✓ All exposure checks passed successfully.")

BENCHMARK PORTFOLIOS — EXPOSURE SUMMARY


portfolio,model,long_exp,short_exp,gross_exp,net_exp
long_only_equal_weight,Random Forest,1.000000,0.000000,1.000000,1.000000
long_only_equal_weight,Ridge,1.000000,0.000000,1.000000,1.000000
long_only_equal_weight,XGBoost,1.000000,0.000000,1.000000,1.000000
long_short_equal_weight,Random Forest,0.500000,-0.500000,1.000000,0.000000
long_short_equal_weight,Ridge,0.500000,-0.500000,1.000000,0.000000
long_short_equal_weight,XGBoost,0.500000,-0.500000,1.000000,0.000000


✓ All exposure checks passed successfully.


Las carteras de referencia quedan validadas con total precisión matemática en los tres modelos. La estrategia Long-Only mantiene una exposición neta constante del 100% en el decil superior ($D10$), mientras que la Long-Short logra una neutralidad de mercado perfecta ($E_{\text{net}} = 0.00$) con un 50% de exposición por lado ($D10$ vs $D1$). Sin sesgos de escala ni diferencias de exposición entre algoritmos, el marco queda listo para pasar a la Sección 5.

## 5. Portfolio Selection Sensitivity

### 5.1 Top 10% vs. Top 20% vs. Top 30%

Se analiza cómo cambia la composición de las carteras long-only al ampliar progresivamente el universo de activos seleccionados. Manteniendo el mismo esquema de Equal Weight, se comparan los percentiles superiores del 10%, 20% y 30% para evaluar el compromiso entre concentración en las señales más extremas y diversificación de la cartera.

In [23]:
# =============================================================================
# Top 10% vs. Top 20% vs. Top 30% — Equal Weight
# =============================================================================

SELECTION_LEVELS = {
    "top_10": 10,
    "top_20": 9,
    "top_30": 8,
}

long_only_selection_weights_list = []


for model, quantile_column in QUANTILE_COLUMNS.items():

    for portfolio_name, minimum_quantile in SELECTION_LEVELS.items():

        # ---------------------------------------------------------------------
        # Select top percentile
        # ---------------------------------------------------------------------

        selected = (
            quantile_data[quantile_column]
            >= minimum_quantile
        )

        selected_index = (
            quantile_data.index[selected]
        )

        # ---------------------------------------------------------------------
        # Count selected positions per date
        # ---------------------------------------------------------------------

        selected_counts = (
            pd.Series(
                1,
                index=selected_index,
            )
            .groupby(level="date")
            .sum()
        )

        # ---------------------------------------------------------------------
        # Equal weights
        # ---------------------------------------------------------------------

        weights = (
            pd.Series(
                1.0,
                index=selected_index,
            )
            .div(
                selected_counts,
                level="date",
            )
        )

        # ---------------------------------------------------------------------
        # Build model portfolio
        # ---------------------------------------------------------------------

        model_weights = (
            weights
            .rename("weight")
            .reset_index()
        )

        model_weights["model"] = model

        model_weights["portfolio"] = (
            f"long_only_{portfolio_name}_equal_weight"
        )

        long_only_selection_weights_list.append(
            model_weights[
                [
                    "date",
                    "ticker",
                    "model",
                    "portfolio",
                    "weight",
                ]
            ]
        )


# =============================================================================
# Combine models and selection levels
# =============================================================================

long_only_selection_weights = pd.concat(
    long_only_selection_weights_list,
    ignore_index=True,
)

La validación de las carteras se estructura en cuatro bloques independientes, con el objetivo de comprobar progresivamente la correcta construcción de los portfolios. Se verifica su integridad, exposición y esquema de weighting, seguido de la coherencia del número de posiciones con la clasificación por cuantiles y, finalmente, el rango de pesos y la ordenación de la concentración.

In [24]:
# =============================================================================
# Validation: Portfolio Integrity
# =============================================================================

print("=" * 80)
print("PORTFOLIO INTEGRITY")
print("=" * 80)


# =============================================================================
# 1. No negative weights
# =============================================================================

assert (
    long_only_selection_weights["weight"] >= 0
).all(), (
    "Negative weights detected."
)

print("✓ All portfolio weights are non-negative.")


# =============================================================================
# 2. No duplicated observations
# =============================================================================

duplicates = (
    long_only_selection_weights
    .duplicated(
        subset=[
            "date",
            "ticker",
            "model",
            "portfolio",
        ]
    )
    .sum()
)

assert duplicates == 0, (
    f"Found {duplicates} duplicated "
    "date-ticker-model-portfolio observations."
)

print(
    "✓ No duplicated "
    "date-ticker-model-portfolio observations."
)

PORTFOLIO INTEGRITY
✓ All portfolio weights are non-negative.
✓ No duplicated date-ticker-model-portfolio observations.


In [25]:
# =============================================================================
# Validation: Exposure & Weighting
# =============================================================================

print("=" * 80)
print("EXPOSURE & WEIGHTING")
print("=" * 80)


# =============================================================================
# 1. Portfolio exposure
# =============================================================================

portfolio_exposure = (
    long_only_selection_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .sum()
)

assert np.allclose(
    portfolio_exposure.values,
    1.00,
), (
    "Portfolio exposure is not equal to 1.00."
)

print(
    "✓ Portfolio exposure = 1.00 "
    "for all portfolios."
)


# =============================================================================
# 2. Equal-weight validation
# =============================================================================

weight_check = (
    long_only_selection_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .agg(
        min_weight="min",
        max_weight="max",
    )
)

assert np.allclose(
    weight_check["min_weight"],
    weight_check["max_weight"],
), (
    "Weights are not equal within at least "
    "one portfolio."
)

print(
    "✓ Equal weighting confirmed "
    "for all portfolios."
)

EXPOSURE & WEIGHTING
✓ Portfolio exposure = 1.00 for all portfolios.
✓ Equal weighting confirmed for all portfolios.


In [26]:
# =============================================================================
# POSITION COUNT
# =============================================================================

print("=" * 80)
print("POSITION COUNT")
print("=" * 80)


for model, quantile_column in QUANTILE_COLUMNS.items():

    quantiles = quantile_data[quantile_column]

    # -------------------------------------------------------------------------
    # Expected position counts from quantile construction
    # -------------------------------------------------------------------------

    expected_top_10 = (
        quantiles
        .groupby(level="date")
        .apply(lambda x: (x == 10).sum())
    )

    expected_top_20 = (
        quantiles
        .groupby(level="date")
        .apply(lambda x: (x >= 9).sum())
    )

    expected_top_30 = (
        quantiles
        .groupby(level="date")
        .apply(lambda x: (x >= 8).sum())
    )

    # -------------------------------------------------------------------------
    # Actual position counts
    # -------------------------------------------------------------------------

    actual_top_10 = (
        long_only_selection_weights[
            (long_only_selection_weights["model"] == model)
            & (
                long_only_selection_weights["portfolio"]
                == "long_only_top_10_equal_weight"
            )
        ]
        .groupby("date")["ticker"]
        .nunique()
    )

    actual_top_20 = (
        long_only_selection_weights[
            (long_only_selection_weights["model"] == model)
            & (
                long_only_selection_weights["portfolio"]
                == "long_only_top_20_equal_weight"
            )
        ]
        .groupby("date")["ticker"]
        .nunique()
    )

    actual_top_30 = (
        long_only_selection_weights[
            (long_only_selection_weights["model"] == model)
            & (
                long_only_selection_weights["portfolio"]
                == "long_only_top_30_equal_weight"
            )
        ]
        .groupby("date")["ticker"]
        .nunique()
    )

    # -------------------------------------------------------------------------
    # Validation
    # -------------------------------------------------------------------------

    assert actual_top_10.equals(expected_top_10), (
        f"Position counts for {model} Top 10% "
        "are inconsistent with the expected selection."
    )

    assert actual_top_20.equals(expected_top_20), (
        f"Position counts for {model} Top 20% "
        "are inconsistent with the expected selection."
    )

    assert actual_top_30.equals(expected_top_30), (
        f"Position counts for {model} Top 30% "
        "are inconsistent with the expected selection."
    )

    # -------------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------------

    print(
        f"✓ {model}: "
        f"Top 10% = {actual_top_10.min()}–{actual_top_10.max()} | "
        f"Top 20% = {actual_top_20.min()}–{actual_top_20.max()} | "
        f"Top 30% = {actual_top_30.min()}–{actual_top_30.max()}"
    )

POSITION COUNT
✓ Ridge: Top 10% = 50–50 | Top 20% = 99–100 | Top 30% = 149–149
✓ XGBoost: Top 10% = 50–50 | Top 20% = 99–100 | Top 30% = 149–149
✓ Random Forest: Top 10% = 50–50 | Top 20% = 99–100 | Top 30% = 149–149


In [27]:
# =============================================================================
# Validation: Weight Range & Selection Ordering
# =============================================================================

print("=" * 80)
print("WEIGHT RANGE & SELECTION ORDERING")
print("=" * 80)


# =============================================================================
# 1. Weight range
# =============================================================================

for portfolio in [
    "long_only_top_10_equal_weight",
    "long_only_top_20_equal_weight",
    "long_only_top_30_equal_weight",
]:

    weights = long_only_selection_weights.loc[
        long_only_selection_weights["portfolio"] == portfolio,
        "weight",
    ]

    print(
        f"✓ {portfolio}: "
        f"Weight range = "
        f"[{weights.min():.6f}, {weights.max():.6f}]"
    )


# =============================================================================
# 2. Position count ordering
# =============================================================================

position_counts = (
    long_only_selection_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["ticker"]
    .nunique()
)

position_count_table = (
    position_counts
    .unstack("portfolio")
)

assert (
    position_count_table[
        "long_only_top_10_equal_weight"
    ]
    <=
    position_count_table[
        "long_only_top_20_equal_weight"
    ]
).all(), (
    "Top 10% has more positions than Top 20%."
)

assert (
    position_count_table[
        "long_only_top_20_equal_weight"
    ]
    <=
    position_count_table[
        "long_only_top_30_equal_weight"
    ]
).all(), (
    "Top 20% has more positions than Top 30%."
)

print(
    "✓ Position counts increase consistently "
    "from Top 10% → Top 20% → Top 30%."
)

WEIGHT RANGE & SELECTION ORDERING
✓ long_only_top_10_equal_weight: Weight range = [0.020000, 0.020000]
✓ long_only_top_20_equal_weight: Weight range = [0.010000, 0.010101]
✓ long_only_top_30_equal_weight: Weight range = [0.006711, 0.006711]
✓ Position counts increase consistently from Top 10% → Top 20% → Top 30%.



### 5.2 Long-short Quantile Sensitivity

Se estudia la sensibilidad de las carteras long-short al tamaño de los extremos seleccionados. Para ello, se comparan estrategias que toman posiciones largas y cortas en los extremos del 10%, 20% y 30% de la distribución, manteniendo constante la exposición bruta y la neutralidad de la exposición neta.


In [28]:
# =============================================================================
# Long-short Quantile Sensitivity — Equal Weight
# =============================================================================

SELECTION_LEVELS = {
    "top_10": {
        "long_min_quantile": 10,
        "short_max_quantile": 1,
    },
    "top_20": {
        "long_min_quantile": 9,
        "short_max_quantile": 2,
    },
    "top_30": {
        "long_min_quantile": 8,
        "short_max_quantile": 3,
    },
}

long_short_selection_weights_list = []


for model, quantile_column in QUANTILE_COLUMNS.items():

    for portfolio_name, selection in SELECTION_LEVELS.items():

        # ---------------------------------------------------------------------
        # Select long and short sides
        # ---------------------------------------------------------------------

        selected_long = (
            quantile_data[quantile_column]
            >= selection["long_min_quantile"]
        )

        selected_short = (
            quantile_data[quantile_column]
            <= selection["short_max_quantile"]
        )

        selected_long_index = (
            quantile_data.index[selected_long]
        )

        selected_short_index = (
            quantile_data.index[selected_short]
        )

        # ---------------------------------------------------------------------
        # Count selected positions per date
        # ---------------------------------------------------------------------

        selected_long_counts = (
            pd.Series(
                1,
                index=selected_long_index,
            )
            .groupby(level="date")
            .sum()
        )

        selected_short_counts = (
            pd.Series(
                1,
                index=selected_short_index,
            )
            .groupby(level="date")
            .sum()
        )

        # ---------------------------------------------------------------------
        # Equal weights
        # ---------------------------------------------------------------------

        long_weights = (
            pd.Series(
                TARGET_LONG_EXPOSURE,
                index=selected_long_index,
            )
            .div(
                selected_long_counts,
                level="date",
            )
        )

        short_weights = (
            pd.Series(
                TARGET_SHORT_EXPOSURE,
                index=selected_short_index,
            )
            .div(
                selected_short_counts,
                level="date",
            )
        )

        # ---------------------------------------------------------------------
        # Combine long and short sides
        # ---------------------------------------------------------------------

        weights = pd.concat(
            [
                long_weights,
                short_weights,
            ]
        ).sort_index()

        # ---------------------------------------------------------------------
        # Build model portfolio
        # ---------------------------------------------------------------------

        model_weights = (
            weights
            .rename("weight")
            .reset_index()
        )

        model_weights["model"] = model

        model_weights["portfolio"] = (
            f"long_short_{portfolio_name}_equal_weight"
        )

        long_short_selection_weights_list.append(
            model_weights[
                [
                    "date",
                    "ticker",
                    "model",
                    "portfolio",
                    "weight",
                ]
            ]
        )


# =============================================================================
# Combine models and selection levels
# =============================================================================

long_short_selection_weights = pd.concat(
    long_short_selection_weights_list,
    ignore_index=True,
)

La validación de las carteras long-short se estructura en cuatro bloques independientes, adaptados a la presencia simultánea de posiciones largas y cortas. Se comprueba su integridad, exposición y esquema de weighting, incluyendo la neutralidad de la exposición neta y la constancia de la exposición bruta. Finalmente, se verifica la coherencia del número de posiciones y del rango de pesos con los distintos niveles de selección y la correcta ordenación de la concentración.

In [29]:
# =============================================================================
# Validation: Portfolio Integrity
# =============================================================================

print("=" * 80)
print("PORTFOLIO INTEGRITY")
print("=" * 80)

# =============================================================================
# 1. No negative weights
# =============================================================================

assert (
    long_only_selection_weights["weight"] >= 0
).all(), (
    "Negative weights detected."
)

print("✓ All portfolio weights are non-negative.")


# =============================================================================
# 2. No duplicated observations
# =============================================================================

duplicates = (
    long_only_selection_weights
    .duplicated(
        subset=[
            "date",
            "ticker",
            "model",
            "portfolio",
        ]
    )
    .sum()
)

assert duplicates == 0, (
    f"Found {duplicates} duplicated "
    "date-ticker-model-portfolio observations."
)

print(
    "✓ No duplicated "
    "date-ticker-model-portfolio observations."
)

PORTFOLIO INTEGRITY
✓ All portfolio weights are non-negative.
✓ No duplicated date-ticker-model-portfolio observations.


In [30]:
# =============================================================================
# Validation: Exposure & Weighting
# =============================================================================

print("=" * 80)
print("EXPOSURE & WEIGHTING")
print("=" * 80)


# =============================================================================
# 1. Long and short exposure
# =============================================================================

long_exposure = (
    long_short_selection_weights[
        long_short_selection_weights["weight"] > 0
    ]
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .sum()
)

short_exposure = (
    long_short_selection_weights[
        long_short_selection_weights["weight"] < 0
    ]
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .sum()
)

assert np.allclose(
    long_exposure.values,
    0.50,
), (
    "Long exposure is not equal to +0.50."
)

assert np.allclose(
    short_exposure.values,
    -0.50,
), (
    "Short exposure is not equal to -0.50."
)

print(
    "✓ Side exposures confirmed: "
    "Long = +0.50 | Short = -0.50."
)


# =============================================================================
# 2. Gross and net exposure
# =============================================================================

gross_exposure = (
    long_short_selection_weights
    .assign(
        abs_weight=(
            long_short_selection_weights["weight"]
            .abs()
        )
    )
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["abs_weight"]
    .sum()
)

net_exposure = (
    long_short_selection_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .sum()
)

assert np.allclose(
    gross_exposure.values,
    1.00,
), (
    "Gross exposure is not equal to 1.00."
)

assert np.allclose(
    net_exposure.values,
    0.00,
    atol=1e-8,
), (
    "Net exposure is not equal to 0.00."
)

print(
    "✓ Portfolio exposure metrics confirmed: "
    "Gross = 1.00 | Net = 0.00."
)


# =============================================================================
# 3. Equal-weight validation
# =============================================================================

long_weight_check = (
    long_short_selection_weights[
        long_short_selection_weights["weight"] > 0
    ]
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .agg(
        min_weight="min",
        max_weight="max",
    )
)

short_weight_check = (
    long_short_selection_weights[
        long_short_selection_weights["weight"] < 0
    ]
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .agg(
        min_weight="min",
        max_weight="max",
    )
)

assert np.allclose(
    long_weight_check["min_weight"],
    long_weight_check["max_weight"],
), (
    "Long-side weights are not equal within at least "
    "one portfolio."
)

assert np.allclose(
    short_weight_check["min_weight"],
    short_weight_check["max_weight"],
), (
    "Short-side weights are not equal within at least "
    "one portfolio."
)

print(
    "✓ Equal weighting confirmed "
    "for both Long and Short sides."
)

EXPOSURE & WEIGHTING
✓ Side exposures confirmed: Long = +0.50 | Short = -0.50.
✓ Portfolio exposure metrics confirmed: Gross = 1.00 | Net = 0.00.
✓ Equal weighting confirmed for both Long and Short sides.


In [31]:
# =============================================================================
# POSITION COUNT
# =============================================================================

print("=" * 80)
print("POSITION COUNT")
print("=" * 80)


for model, quantile_column in QUANTILE_COLUMNS.items():

    quantiles = quantile_data[quantile_column]

    # -------------------------------------------------------------------------
    # Expected position counts from quantile construction
    # -------------------------------------------------------------------------

    expected_long_top_10 = (
        quantiles
        .groupby(level="date")
        .apply(lambda x: (x == 10).sum())
    )

    expected_short_top_10 = (
        quantiles
        .groupby(level="date")
        .apply(lambda x: (x == 1).sum())
    )

    expected_long_top_20 = (
        quantiles
        .groupby(level="date")
        .apply(lambda x: (x >= 9).sum())
    )

    expected_short_top_20 = (
        quantiles
        .groupby(level="date")
        .apply(lambda x: (x <= 2).sum())
    )

    expected_long_top_30 = (
        quantiles
        .groupby(level="date")
        .apply(lambda x: (x >= 8).sum())
    )

    expected_short_top_30 = (
        quantiles
        .groupby(level="date")
        .apply(lambda x: (x <= 3).sum())
    )

    # -------------------------------------------------------------------------
    # Actual position counts
    # -------------------------------------------------------------------------

    model_weights = (
        long_short_selection_weights[
            long_short_selection_weights["model"] == model
        ]
    )

    # Long positions
    actual_long_top_10 = (
        model_weights[
            (model_weights["portfolio"]
             == "long_short_top_10_equal_weight")
            & (model_weights["weight"] > 0)
        ]
        .groupby("date")["ticker"]
        .nunique()
    )

    actual_long_top_20 = (
        model_weights[
            (model_weights["portfolio"]
             == "long_short_top_20_equal_weight")
            & (model_weights["weight"] > 0)
        ]
        .groupby("date")["ticker"]
        .nunique()
    )

    actual_long_top_30 = (
        model_weights[
            (model_weights["portfolio"]
             == "long_short_top_30_equal_weight")
            & (model_weights["weight"] > 0)
        ]
        .groupby("date")["ticker"]
        .nunique()
    )

    # Short positions
    actual_short_top_10 = (
        model_weights[
            (model_weights["portfolio"]
             == "long_short_top_10_equal_weight")
            & (model_weights["weight"] < 0)
        ]
        .groupby("date")["ticker"]
        .nunique()
    )

    actual_short_top_20 = (
        model_weights[
            (model_weights["portfolio"]
             == "long_short_top_20_equal_weight")
            & (model_weights["weight"] < 0)
        ]
        .groupby("date")["ticker"]
        .nunique()
    )

    actual_short_top_30 = (
        model_weights[
            (model_weights["portfolio"]
             == "long_short_top_30_equal_weight")
            & (model_weights["weight"] < 0)
        ]
        .groupby("date")["ticker"]
        .nunique()
    )

    # -------------------------------------------------------------------------
    # Validation
    # -------------------------------------------------------------------------

    assert actual_long_top_10.equals(expected_long_top_10)
    assert actual_short_top_10.equals(expected_short_top_10)

    assert actual_long_top_20.equals(expected_long_top_20)
    assert actual_short_top_20.equals(expected_short_top_20)

    assert actual_long_top_30.equals(expected_long_top_30)
    assert actual_short_top_30.equals(expected_short_top_30)

    # -------------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------------

    print(
        f"✓ {model}: "
        f"Top 10% → "
        f"Long = {actual_long_top_10.min()}–{actual_long_top_10.max()} | "
        f"Short = {actual_short_top_10.min()}–{actual_short_top_10.max()}"
    )

    print(
        f"  {model}: "
        f"Top 20% → "
        f"Long = {actual_long_top_20.min()}–{actual_long_top_20.max()} | "
        f"Short = {actual_short_top_20.min()}–{actual_short_top_20.max()}"
    )

    print(
        f"  {model}: "
        f"Top 30% → "
        f"Long = {actual_long_top_30.min()}–{actual_long_top_30.max()} | "
        f"Short = {actual_short_top_30.min()}–{actual_short_top_30.max()}"
    )


print("\n✓ Position counts are consistent with quantile construction.")

POSITION COUNT
✓ Ridge: Top 10% → Long = 50–50 | Short = 49–49
  Ridge: Top 20% → Long = 99–100 | Short = 98–99
  Ridge: Top 30% → Long = 149–149 | Short = 148–148
✓ XGBoost: Top 10% → Long = 50–50 | Short = 49–49
  XGBoost: Top 20% → Long = 99–100 | Short = 98–99
  XGBoost: Top 30% → Long = 149–149 | Short = 148–148
✓ Random Forest: Top 10% → Long = 50–50 | Short = 49–49
  Random Forest: Top 20% → Long = 99–100 | Short = 98–99
  Random Forest: Top 30% → Long = 149–149 | Short = 148–148

✓ Position counts are consistent with quantile construction.


In [32]:
# =============================================================================
# Validation: Weight Range & Selection Ordering
# =============================================================================

print("=" * 80)
print("WEIGHT RANGE & SELECTION ORDERING")
print("=" * 80)


# =============================================================================
# 1. Weight range
# =============================================================================

for portfolio in [
    "long_short_top_10_equal_weight",
    "long_short_top_20_equal_weight",
    "long_short_top_30_equal_weight",
]:

    long_weights = (
        long_short_selection_weights.loc[
            (
                long_short_selection_weights["portfolio"]
                == portfolio
            )
            & (
                long_short_selection_weights["weight"]
                > 0
            ),
            "weight",
        ]
    )

    short_weights = (
        long_short_selection_weights.loc[
            (
                long_short_selection_weights["portfolio"]
                == portfolio
            )
            & (
                long_short_selection_weights["weight"]
                < 0
            ),
            "weight",
        ]
    )

    print(
        f"✓ {portfolio}: "
        f"Long = [{long_weights.min():.6f}, "
        f"{long_weights.max():.6f}] | "
        f"Short = [{short_weights.min():.6f}, "
        f"{short_weights.max():.6f}]"
    )


# =============================================================================
# 2. Position count ordering
# =============================================================================

long_position_counts = (
    long_short_selection_weights[
        long_short_selection_weights["weight"] > 0
    ]
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["ticker"]
    .nunique()
)

short_position_counts = (
    long_short_selection_weights[
        long_short_selection_weights["weight"] < 0
    ]
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["ticker"]
    .nunique()
)


long_position_count_table = (
    long_position_counts
    .unstack("portfolio")
)

short_position_count_table = (
    short_position_counts
    .unstack("portfolio")
)


# =============================================================================
# 3. Long-side ordering
# =============================================================================

assert (
    long_position_count_table[
        "long_short_top_10_equal_weight"
    ]
    <=
    long_position_count_table[
        "long_short_top_20_equal_weight"
    ]
).all(), (
    "Long Top 10% has more positions than "
    "Long Top 20%."
)

assert (
    long_position_count_table[
        "long_short_top_20_equal_weight"
    ]
    <=
    long_position_count_table[
        "long_short_top_30_equal_weight"
    ]
).all(), (
    "Long Top 20% has more positions than "
    "Long Top 30%."
)


# =============================================================================
# 4. Short-side ordering
# =============================================================================

assert (
    short_position_count_table[
        "long_short_top_10_equal_weight"
    ]
    <=
    short_position_count_table[
        "long_short_top_20_equal_weight"
    ]
).all(), (
    "Short Top 10% has more positions than "
    "Short Top 20%."
)

assert (
    short_position_count_table[
        "long_short_top_20_equal_weight"
    ]
    <=
    short_position_count_table[
        "long_short_top_30_equal_weight"
    ]
).all(), (
    "Short Top 20% has more positions than "
    "Short Top 30%."
)


print(
    "✓ Position counts increase consistently "
    "from Top 10% → Top 20% → Top 30% "
    "on both Long and Short sides."
)

WEIGHT RANGE & SELECTION ORDERING
✓ long_short_top_10_equal_weight: Long = [0.010000, 0.010000] | Short = [-0.010204, -0.010204]
✓ long_short_top_20_equal_weight: Long = [0.005000, 0.005051] | Short = [-0.005102, -0.005051]
✓ long_short_top_30_equal_weight: Long = [0.003356, 0.003356] | Short = [-0.003378, -0.003378]
✓ Position counts increase consistently from Top 10% → Top 20% → Top 30% on both Long and Short sides.



### 5.3 Position Count & Concentration

Se analiza el número de posiciones resultante de cada nivel de selección y su relación con la concentración de la cartera. Este análisis permite cuantificar cómo la ampliación del universo seleccionado modifica el número de activos y, bajo Equal Weight, el peso individual asignado a cada posición.

In [33]:
# =============================================================================
# Position Count & Concentration — Long-only
# =============================================================================

print("=" * 80)
print("LONG-ONLY — POSITION COUNT & CONCENTRATION")
print("=" * 80)


# =============================================================================
# 1. Daily position count
# =============================================================================

long_only_position_counts = (
    long_only_selection_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["ticker"]
    .nunique()
)


# =============================================================================
# 2. Position count range
# =============================================================================

long_only_position_summary = (
    long_only_position_counts
    .groupby(
        [
            "model",
            "portfolio",
        ]
    )
    .agg(
        min_positions="min",
        max_positions="max",
    )
    .reset_index()
)


# =============================================================================
# 3. Weight range
# =============================================================================

long_only_weight_summary = (
    long_only_selection_weights
    .groupby(
        [
            "model",
            "portfolio",
        ]
    )["weight"]
    .agg(
        min_weight="min",
        max_weight="max",
    )
    .reset_index()
)


# =============================================================================
# 4. Combine summaries
# =============================================================================

long_only_summary = (
    long_only_position_summary
    .merge(
        long_only_weight_summary,
        on=[
            "model",
            "portfolio",
        ],
    )
)


# =============================================================================
# 5. Display results
# =============================================================================

for _, row in long_only_summary.iterrows():

    print(
        f"{row['model']} | "
        f"{row['portfolio']} | "
        f"Positions = "
        f"{int(row['min_positions'])}"
        f"–{int(row['max_positions'])} | "
        f"Weight = "
        f"{row['min_weight']:.6f}"
        f"–{row['max_weight']:.6f}"
    )


print("\n" + "=" * 80)
print("LONG-ONLY SUMMARY COMPLETED")
print("=" * 80)

LONG-ONLY — POSITION COUNT & CONCENTRATION
Random Forest | long_only_top_10_equal_weight | Positions = 50–50 | Weight = 0.020000–0.020000
Random Forest | long_only_top_20_equal_weight | Positions = 99–100 | Weight = 0.010000–0.010101
Random Forest | long_only_top_30_equal_weight | Positions = 149–149 | Weight = 0.006711–0.006711
Ridge | long_only_top_10_equal_weight | Positions = 50–50 | Weight = 0.020000–0.020000
Ridge | long_only_top_20_equal_weight | Positions = 99–100 | Weight = 0.010000–0.010101
Ridge | long_only_top_30_equal_weight | Positions = 149–149 | Weight = 0.006711–0.006711
XGBoost | long_only_top_10_equal_weight | Positions = 50–50 | Weight = 0.020000–0.020000
XGBoost | long_only_top_20_equal_weight | Positions = 99–100 | Weight = 0.010000–0.010101
XGBoost | long_only_top_30_equal_weight | Positions = 149–149 | Weight = 0.006711–0.006711

LONG-ONLY SUMMARY COMPLETED


In [34]:
# =============================================================================
# Position Count & Concentration — Long-short
# =============================================================================

print("=" * 80)
print("LONG-SHORT — POSITION COUNT & CONCENTRATION")
print("=" * 80)


# =============================================================================
# 1. Add portfolio side
# =============================================================================

long_short_data = (
    long_short_selection_weights
    .assign(
        side=np.where(
            long_short_selection_weights["weight"] > 0,
            "Long",
            "Short",
        )
    )
)


# =============================================================================
# 2. Daily position count
# =============================================================================

long_short_position_counts = (
    long_short_data
    .groupby(
        [
            "date",
            "model",
            "portfolio",
            "side",
        ]
    )["ticker"]
    .nunique()
)


# =============================================================================
# 3. Position count range
# =============================================================================

long_short_position_summary = (
    long_short_position_counts
    .groupby(
        [
            "model",
            "portfolio",
            "side",
        ]
    )
    .agg(
        min_positions="min",
        max_positions="max",
    )
    .reset_index()
)


# =============================================================================
# 4. Weight range
# =============================================================================

long_short_weight_summary = (
    long_short_data
    .groupby(
        [
            "model",
            "portfolio",
            "side",
        ]
    )["weight"]
    .agg(
        min_weight="min",
        max_weight="max",
    )
    .reset_index()
)


# =============================================================================
# 5. Combine summaries
# =============================================================================

long_short_summary = (
    long_short_position_summary
    .merge(
        long_short_weight_summary,
        on=[
            "model",
            "portfolio",
            "side",
        ],
    )
)


# =============================================================================
# 6. Display results
# =============================================================================

for _, row in long_short_summary.iterrows():

    print(
        f"{row['model']} | "
        f"{row['portfolio']} | "
        f"{row['side']} | "
        f"Positions = "
        f"{int(row['min_positions'])}"
        f"–{int(row['max_positions'])} | "
        f"Weight = "
        f"{row['min_weight']:.6f}"
        f"–{row['max_weight']:.6f}"
    )


print("\n" + "=" * 80)
print("LONG-SHORT SUMMARY COMPLETED")
print("=" * 80)

LONG-SHORT — POSITION COUNT & CONCENTRATION
Random Forest | long_short_top_10_equal_weight | Long | Positions = 50–50 | Weight = 0.010000–0.010000
Random Forest | long_short_top_10_equal_weight | Short | Positions = 49–49 | Weight = -0.010204–-0.010204
Random Forest | long_short_top_20_equal_weight | Long | Positions = 99–100 | Weight = 0.005000–0.005051
Random Forest | long_short_top_20_equal_weight | Short | Positions = 98–99 | Weight = -0.005102–-0.005051
Random Forest | long_short_top_30_equal_weight | Long | Positions = 149–149 | Weight = 0.003356–0.003356
Random Forest | long_short_top_30_equal_weight | Short | Positions = 148–148 | Weight = -0.003378–-0.003378
Ridge | long_short_top_10_equal_weight | Long | Positions = 50–50 | Weight = 0.010000–0.010000
Ridge | long_short_top_10_equal_weight | Short | Positions = 49–49 | Weight = -0.010204–-0.010204
Ridge | long_short_top_20_equal_weight | Long | Positions = 99–100 | Weight = 0.005000–0.005051
Ridge | long_short_top_20_equal_wei

Los resultados muestran un comportamiento coherente con el esquema de selección y asignación Equal Weight. A medida que se amplía el universo seleccionado del Top 10% al Top 30%, aumenta el número de posiciones y disminuye el peso individual de cada activo, reduciendo la concentración de la cartera. En las estrategias long-short, la ligera diferencia en el número de activos entre los extremos long y short se compensa mediante pequeños ajustes en el peso individual, manteniendo constante la exposición objetivo de cada lado.

## 6. Portfolio Weighting

Una vez definida la selección de activos, esta sección estudia distintas metodologías para transformar dicha selección en pesos concretos de cartera. El Equal Weight, establecido previamente como benchmark en la sección 4, sirve como referencia frente a metodologías que incorporan información adicional sobre la intensidad de la señal o el riesgo de los activos.


### 6.1 Heuristic Allocation — Signal Weighting

Los métodos heurísticos permiten transformar la selección de activos en pesos de forma sencilla y robusta, sin necesidad de estimar matrices de covarianzas ni resolver problemas de optimización. El Equal Weight, utilizado como benchmark en la sección 4, establece la referencia frente a la que se evaluarán metodologías de weighting más sofisticadas. En este apartado se introduce Signal Weighting, que asigna mayores pesos a los activos con señales relativamente más fuertes, con el objetivo de incorporar la intensidad de la predicción del modelo en la construcción de la cartera.


Pasamos de Equal Weight a Signal Weighting: en vez de asignar el capital a partes iguales dentro del universo seleccionado, el peso de cada activo será proporcional a la intensidad de la señal del modelo, medida mediante su ranking cross-sectional (prediction__rank). Esta representación es comparable entre activos y modelos y permite establecer un orden determinista mediante method="first".

Una decisión clave antes de implementar la metodología es que el ranking se recalcula dentro del subconjunto seleccionado, en lugar de heredarse del universo completo. Si se utilizara directamente el ranking global, los valores dentro del D10 quedarían comprimidos en un rango muy estrecho (por ejemplo, $0.90$–$1.00$), haciendo que el tilt resultante fuera muy reducido frente a Equal Weight. Al recalcular el ranking dentro del D10, los valores se redistribuyen entre $1/N$ y $1.0$, permitiendo introducir una diferenciación significativa entre las posiciones seleccionadas.

Para long-only, el peso de cada activo se define como:

$$w_{i,t} = \frac{r_{i,t}}{\sum_{j \in S_t} r_{j,t}}$$

Para long-short, el weighting se realiza de forma independiente en cada lado para mantener $E_L = +0.50$ y $E_S = -0.50$: en el lado largo se utiliza directamente $r_{i,t}$, mientras que en el lado corto se utiliza la intensidad bajista $1 - r_{i,t}$, de modo que los activos con menor ranking reciben un mayor peso absoluto.

Se consideró y descartó una alternativa que desplazaba la señal respecto al mínimo o máximo del subgrupo (por ejemplo, $s_i = r_i - r_{\min}$ en el lado largo), con el objetivo de asignar una intensidad cercana a cero al activo situado en el límite del universo seleccionado. Esta transformación fuerza matemáticamente a que dicho activo reciba un peso exactamente igual a cero, reduciendo de forma silenciosa el tamaño efectivo de la cartera y contradiciendo el principio de diversificación establecido en las secciones anteriores. El recálculo del ranking dentro del universo seleccionado permite igualmente penalizar a los activos marginales sin excluirlos de facto de la cartera, ya que el mínimo del ranking normalizado es $1/N$ y nunca alcanza cero por construcción.

In [35]:
# =============================================================================
# Signal Weighting — Long-Only
# =============================================================================

SIGNAL_WEIGHTING_K = 1.0

signal_long_only_weights_list = []


for model, prediction_column in MODEL_COLUMNS.items():

    quantile_column = QUANTILE_COLUMNS[model]

    for portfolio_name, selection in SELECTION_LEVELS.items():

        # ---------------------------------------------------------------------
        # Select portfolio universe
        # ---------------------------------------------------------------------

        selected = (
            quantile_data[quantile_column]
            >= selection["long_min_quantile"]
        )

        selected_index = (
            quantile_data.index[selected]
        )

        # ---------------------------------------------------------------------
        # Retrieve model predictions for selected assets
        # ---------------------------------------------------------------------

        selected_predictions = (
            oos_predictions.loc[
                selected_index,
                prediction_column,
            ]
        )

        # ---------------------------------------------------------------------
        # Re-rank within selected universe
        # ---------------------------------------------------------------------

        selected_ranks = (
            selected_predictions
            .groupby(level="date")
            .rank(
                method="first",
                pct=True,
            )
        )

        # ---------------------------------------------------------------------
        # Apply signal intensity
        # ---------------------------------------------------------------------

        signal_strength = (
            selected_ranks
            ** SIGNAL_WEIGHTING_K
        )

        # ---------------------------------------------------------------------
        # Normalize weights within each date
        # ---------------------------------------------------------------------

        weight_denominator = (
            signal_strength
            .groupby(level="date")
            .transform("sum")
        )

        weights = (
            signal_strength
            / weight_denominator
        )

        # ---------------------------------------------------------------------
        # Build model portfolio
        # ---------------------------------------------------------------------

        model_weights = (
            weights
            .rename("weight")
            .reset_index()
        )

        model_weights["model"] = model

        model_weights["portfolio"] = (
            f"long_only_{portfolio_name}_signal_weight"
        )

        signal_long_only_weights_list.append(
            model_weights[
                [
                    "date",
                    "ticker",
                    "model",
                    "portfolio",
                    "weight",
                ]
            ]
        )


# =============================================================================
# Combine models and selection levels
# =============================================================================

signal_long_only_weights = pd.concat(
    signal_long_only_weights_list,
    ignore_index=True,
)

La validación de las carteras con Signal Weighting se estructura en cuatro bloques, siguiendo una lógica progresiva. Primero se comprueba la integridad estructural de las carteras; después, la exposición y diferenciación de los pesos; a continuación, se verifica que el weighting no altera el número de posiciones seleccionadas; y, finalmente, se analiza el rango de pesos y la concentración, comprobando que el tilt hacia las señales más fuertes aumenta de forma coherente al reducir el universo seleccionado.

In [36]:
# =============================================================================
# Validation: Portfolio Integrity
# =============================================================================

print("=" * 80)
print("PORTFOLIO INTEGRITY")
print("=" * 80)


# =============================================================================
# 1. No negative weights
# =============================================================================

assert (
    signal_long_only_weights["weight"] >= 0
).all(), (
    "Negative weights detected."
)

print("✓ All portfolio weights are non-negative.")


# =============================================================================
# 2. No duplicated observations
# =============================================================================

duplicates = (
    signal_long_only_weights
    .duplicated(
        subset=[
            "date",
            "ticker",
            "model",
            "portfolio",
        ]
    )
    .sum()
)

assert duplicates == 0, (
    f"Found {duplicates} duplicated "
    "date-ticker-model-portfolio observations."
)

print(
    "✓ No duplicated "
    "date-ticker-model-portfolio observations."
)

PORTFOLIO INTEGRITY
✓ All portfolio weights are non-negative.
✓ No duplicated date-ticker-model-portfolio observations.


In [37]:
# =============================================================================
# Validation: Exposure & Signal Weighting
# =============================================================================

print("=" * 80)
print("EXPOSURE & SIGNAL WEIGHTING")
print("=" * 80)


# =============================================================================
# 1. Portfolio exposure
# =============================================================================

portfolio_exposure = (
    signal_long_only_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .sum()
)

assert np.allclose(
    portfolio_exposure.values,
    1.00,
), (
    "Portfolio exposure is not equal to 1.00."
)

print(
    "✓ Portfolio exposure = 1.00 "
    "for all portfolios."
)


# =============================================================================
# 2. Signal differentiation
# =============================================================================

weight_check = (
    signal_long_only_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .agg(
        min_weight="min",
        max_weight="max",
    )
)

assert (
    weight_check["min_weight"]
    < weight_check["max_weight"]
).all(), (
    "Signal weighting is not producing "
    "differentiated weights."
)

print(
    "✓ Signal weighting produces "
    "differentiated weights."
)

EXPOSURE & SIGNAL WEIGHTING
✓ Portfolio exposure = 1.00 for all portfolios.
✓ Signal weighting produces differentiated weights.


In [38]:
# =============================================================================
# Validation: Position Count
# =============================================================================

print("=" * 80)
print("POSITION COUNT")
print("=" * 80)


for model, quantile_column in QUANTILE_COLUMNS.items():

    quantiles = quantile_data[quantile_column]

    # -------------------------------------------------------------------------
    # Expected position counts from quantile construction
    # -------------------------------------------------------------------------

    expected_top_10 = (
        quantiles
        .groupby(level="date")
        .apply(lambda x: (x == 10).sum())
    )

    expected_top_20 = (
        quantiles
        .groupby(level="date")
        .apply(lambda x: (x >= 9).sum())
    )

    expected_top_30 = (
        quantiles
        .groupby(level="date")
        .apply(lambda x: (x >= 8).sum())
    )

    # -------------------------------------------------------------------------
    # Actual position counts
    # -------------------------------------------------------------------------

    actual_top_10 = (
        signal_long_only_weights[
            (signal_long_only_weights["model"] == model)
            & (
                signal_long_only_weights["portfolio"]
                == "long_only_top_10_signal_weight"
            )
        ]
        .groupby("date")["ticker"]
        .nunique()
    )

    actual_top_20 = (
        signal_long_only_weights[
            (signal_long_only_weights["model"] == model)
            & (
                signal_long_only_weights["portfolio"]
                == "long_only_top_20_signal_weight"
            )
        ]
        .groupby("date")["ticker"]
        .nunique()
    )

    actual_top_30 = (
        signal_long_only_weights[
            (signal_long_only_weights["model"] == model)
            & (
                signal_long_only_weights["portfolio"]
                == "long_only_top_30_signal_weight"
            )
        ]
        .groupby("date")["ticker"]
        .nunique()
    )

    # -------------------------------------------------------------------------
    # Validation
    # -------------------------------------------------------------------------

    assert actual_top_10.equals(expected_top_10), (
        f"Position counts for {model} Top 10% "
        "are inconsistent with the expected selection."
    )

    assert actual_top_20.equals(expected_top_20), (
        f"Position counts for {model} Top 20% "
        "are inconsistent with the expected selection."
    )

    assert actual_top_30.equals(expected_top_30), (
        f"Position counts for {model} Top 30% "
        "are inconsistent with the expected selection."
    )

    # -------------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------------

    print(
        f"✓ {model}: "
        f"Top 10% = {actual_top_10.min()}–{actual_top_10.max()} | "
        f"Top 20% = {actual_top_20.min()}–{actual_top_20.max()} | "
        f"Top 30% = {actual_top_30.min()}–{actual_top_30.max()}"
    )

POSITION COUNT
✓ Ridge: Top 10% = 50–50 | Top 20% = 99–100 | Top 30% = 149–149
✓ XGBoost: Top 10% = 50–50 | Top 20% = 99–100 | Top 30% = 149–149
✓ Random Forest: Top 10% = 50–50 | Top 20% = 99–100 | Top 30% = 149–149


In [39]:
# =============================================================================
# Validation: Weight Range & Concentration
# =============================================================================

print("=" * 80)
print("WEIGHT RANGE & CONCENTRATION")
print("=" * 80)


# =============================================================================
# 1. Weight range
# =============================================================================

for portfolio in [
    "long_only_top_10_signal_weight",
    "long_only_top_20_signal_weight",
    "long_only_top_30_signal_weight",
]:

    weights = signal_long_only_weights.loc[
        signal_long_only_weights["portfolio"] == portfolio,
        "weight",
    ]

    print(
        f"✓ {portfolio}: "
        f"Weight range = "
        f"[{weights.min():.6f}, {weights.max():.6f}]"
    )


# =============================================================================
# 2. Concentration ordering
# =============================================================================

max_weights = (
    signal_long_only_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .max()
    .unstack("portfolio")
)

assert (
    max_weights[
        "long_only_top_10_signal_weight"
    ]
    >=
    max_weights[
        "long_only_top_20_signal_weight"
    ]
).all(), (
    "Top 10% does not have greater maximum weight "
    "than Top 20%."
)

assert (
    max_weights[
        "long_only_top_20_signal_weight"
    ]
    >=
    max_weights[
        "long_only_top_30_signal_weight"
    ]
).all(), (
    "Top 20% does not have greater maximum weight "
    "than Top 30%."
)

print(
    "✓ Maximum weights increase consistently "
    "from Top 30% → Top 20% → Top 10%."
)

WEIGHT RANGE & CONCENTRATION
✓ long_only_top_10_signal_weight: Weight range = [0.000784, 0.039216]
✓ long_only_top_20_signal_weight: Weight range = [0.000198, 0.020000]
✓ long_only_top_30_signal_weight: Weight range = [0.000089, 0.013333]
✓ Maximum weights increase consistently from Top 30% → Top 20% → Top 10%.


### 6.2 Risk-Based Allocation

En esta sección se modifica el mecanismo de asignación de pesos utilizado en las carteras benchmark. La señal del modelo continúa utilizándose como mecanismo de selección de activos, pero deja de determinar directamente la intensidad de los pesos. En su lugar, los pesos se obtienen a partir de información sobre el riesgo de los activos seleccionados.

En el enfoque Risk-Based, las predicciones de los modelos de machine learning se emplean exclusivamente para seleccionar el universo de activos en cada fecha, delegando la asignación de pesos a métricas de riesgo observadas en el mercado. De esta forma se busca comprobar si la gestión explícita del riesgo logra mejorar el perfil de rentabilidad-riesgo frente a la ponderación por defecto (Equal Weight), sin modificar la selección producida por el modelo.

Para estimar el riesgo se calcula una volatilidad rolling de corto plazo de 21 sesiones de trading ($\approx 1$ mes) a partir de los retornos diarios simples:

$$\sigma_{i,t} = \text{Std}(r_{i,t-20}, \dots, r_{i,t-1})$$

Esta ventana permite reaccionar con agilidad a cambios de régimen en el mercado, proporcionando un compromiso entre actualidad de la estimación y disponibilidad de observaciones. La estimación utiliza estrictamente información disponible antes de la decisión de inversión en la fecha $t$, garantizando la ausencia de look-ahead bias. Además, para poder calcular las métricas desde el primer día del período fuera de muestra (OOS, iniciado el 15 de enero de 2025), se emplean las 21 sesiones de mercado inmediatamente anteriores procedentes del período buffer previo a test.

Dentro de este marco se implementan dos metodologías:

- **Inverse Volatility**: Asigna a cada activo un peso inversamente proporcional a su volatilidad individual reciente ($w_{i,t} \propto 1/\sigma_{i,t}$):

$$w_{i,t} = \frac{1/\sigma_{i,t}}{\sum_{j \in S_t} 1/\sigma_{j,t}}$$

De este modo, los valores más estables reciben una mayor asignación de capital mientras que los más volátiles se penalizan. Se utiliza la volatilidad total en lugar de la downside volatility para mantener una definición neutra de riesgo que no introduzca hipótesis adicionales sobre la asimetría de los retornos, reservando esta última para posibles análisis de robustez.

- **Risk Parity**: Pasa del riesgo individual al análisis del riesgo conjunto de la cartera mediante la matriz de covarianzas $\boldsymbol{\Sigma}_t$. Dado que el número de activos seleccionados ($50$ a $150$) supera las $21$ observaciones de la ventana temporal, la matriz muestral resulta inestable y ruidosa. Para resolverlo, se aplica la regularización de Ledoit-Wolf Shrinkage, que combina la covarianza observada ($\boldsymbol{\Sigma}_{\text{sample}}$) con una matriz objetivo estructurada ($\boldsymbol{\Sigma}_{\text{target}}$):

$$\boldsymbol{\Sigma}_{\text{LW}} = (1 - \lambda)\boldsymbol{\Sigma}_{\text{sample}} + \lambda\boldsymbol{\Sigma}_{\text{target}}$$

La intensidad de suavizado $\lambda$ se estima automáticamente mediante el procedimiento de Ledoit-Wolf.. A partir de esta matriz regularizada, la contribución total al riesgo ($RC_i$) de un activo en una cartera con pesos $\mathbf{w}$ y volatilidad $\sigma_p = \sqrt{\mathbf{w}^\top \boldsymbol{\Sigma}_{\text{LW}} \mathbf{w}}$ viene dada por:

$$RC_i = w_i \cdot \frac{(\boldsymbol{\Sigma}_{\text{LW}} \mathbf{w})_i}{\sigma_p}$$

El algoritmo de Risk Parity pasa del análisis del riesgo individual al riesgo conjunto de la cartera mediante la matriz de covarianzas $\boldsymbol{\Sigma}_t$. El objetivo no es asignar el mismo capital a cada activo, sino conseguir que cada posición contribuya aproximadamente en la misma proporción al riesgo total de la cartera ($RC_1 = RC_2 = \dots = RC_N$). Por ello, un activo con mayor volatilidad o mayor exposición al riesgo conjunto puede recibir un peso inferior a otro activo más estable o menos correlacionado con el resto de posiciones.

#### 6.2.1 Inverse Volatility

La volatilidad utilizada para la asignación de pesos es conceptualmente similar a la rolling volatility construida previamente como factor, pero no constituye exactamente la misma variable. En la fase de feature engineering se calculó una volatilidad rolling de 252 sesiones, diseñada para capturar una medida de riesgo de medio-largo plazo y utilizarse como predictor del modelo. 

En cambio, para Inverse Volatility se emplea una ventana de 21 sesiones, con el objetivo de obtener una estimación de riesgo reciente que pueda adaptarse rápidamente a cambios en las condiciones de mercado. Además, en esta fase la estimación se desplaza una sesión (shift(1)) para garantizar que únicamente utiliza información disponible antes de la formación de la cartera y evitar cualquier look-ahead bias.

Por tanto, la elección de 21 sesiones no busca replicar el factor de volatilidad utilizado por los modelos, sino proporcionar una medida independiente y contemporánea del riesgo utilizada exclusivamente para la asignación de pesos.

In [40]:
# =============================================================================
# Inverse Volatility — Risk-Based Allocation
# =============================================================================

VOLATILITY_WINDOW = 21

SELECTION_LEVELS = {
    "top_10": 10,
    "top_20": 9,
    "top_30": 8,
}

inverse_volatility_weights_list = []


# =============================================================================
#  Rolling volatility
# =============================================================================

rolling_volatility = (
    log_returns
    .shift(1)
    .rolling(
        window=VOLATILITY_WINDOW,
        min_periods=VOLATILITY_WINDOW,
    )
    .std()
)


# =============================================================================
#  Build portfolios
# =============================================================================

for model, quantile_column in QUANTILE_COLUMNS.items():

    for portfolio_name, minimum_quantile in SELECTION_LEVELS.items():

        # ---------------------------------------------------------------------
        # Select top percentile
        # ---------------------------------------------------------------------

        selected = (
            quantile_data[quantile_column]
            >= minimum_quantile
        )

        selected_index = (
            quantile_data.index[selected]
        )

        # ---------------------------------------------------------------------
        # Extract date and ticker
        # ---------------------------------------------------------------------

        selected_dates = selected_index.get_level_values("date")
        selected_tickers = selected_index.get_level_values("ticker")

        selected_volatility = pd.Series(
            rolling_volatility
            .stack()
            .reindex(
                pd.MultiIndex.from_arrays(
                    [
                        selected_dates,
                        selected_tickers,
                    ],
                    names=["date", "ticker"],
                )
            )
            .values,
            index=selected_index,
        )

        # ---------------------------------------------------------------------
        # Remove observations without sufficient volatility history
        # ---------------------------------------------------------------------

        valid_volatility = (
            selected_volatility.notna()
            & (selected_volatility > 0)
        )

        selected_volatility = (
            selected_volatility[valid_volatility]
        )

        # ---------------------------------------------------------------------
        # Inverse volatility signal
        # ---------------------------------------------------------------------

        inverse_volatility = (
            1.0 / selected_volatility
        )

        # ---------------------------------------------------------------------
        # Normalize weights within each date
        # ---------------------------------------------------------------------

        inverse_volatility_sum = (
            inverse_volatility
            .groupby(level="date")
            .transform("sum")
        )

        weights = (
            inverse_volatility
            .div(inverse_volatility_sum)
        )

        # ---------------------------------------------------------------------
        # Build model portfolio
        # ---------------------------------------------------------------------

        model_weights = (
            weights
            .rename("weight")
            .reset_index()
        )

        model_weights["model"] = model

        model_weights["portfolio"] = (
            f"long_only_{portfolio_name}_inverse_volatility"
        )

        inverse_volatility_weights_list.append(
            model_weights[
                [
                    "date",
                    "ticker",
                    "model",
                    "portfolio",
                    "weight",
                ]
            ]
        )


# =============================================================================
# Combine models and selection levels
# =============================================================================

inverse_volatility_weights = pd.concat(
    inverse_volatility_weights_list,
    ignore_index=True,
)

La validación de Inverse Volatility se estructura en cuatro bloques destinados a comprobar tanto la integridad de las carteras como la correcta aplicación del esquema de ponderación. Se verifica la ausencia de pesos inválidos o duplicados, la exposición total de las carteras y la coherencia del número de posiciones con la selección por cuantiles. Finalmente, se comprueba que los pesos obtenidos coinciden con la fórmula de asignación inversamente proporcional a la volatilidad, validando directamente la lógica del método.

In [41]:
# =============================================================================
# Validation: Portfolio Integrity
# =============================================================================

print("=" * 80)
print("PORTFOLIO INTEGRITY")
print("=" * 80)


# =============================================================================
# 1. No negative weights
# =============================================================================

assert (
    inverse_volatility_weights["weight"] > 0
).all(), (
    "Non-positive weights detected."
)

print("✓ All portfolio weights are strictly positive.")


# =============================================================================
# 2. No duplicated observations
# =============================================================================

duplicates = (
    inverse_volatility_weights
    .duplicated(
        subset=[
            "date",
            "ticker",
            "model",
            "portfolio",
        ]
    )
    .sum()
)

assert duplicates == 0, (
    f"Found {duplicates} duplicated "
    "date-ticker-model-portfolio observations."
)

print(
    "✓ No duplicated "
    "date-ticker-model-portfolio observations."
)

PORTFOLIO INTEGRITY
✓ All portfolio weights are strictly positive.
✓ No duplicated date-ticker-model-portfolio observations.


In [42]:
# =============================================================================
# Validation: Exposure & Weighting
# =============================================================================

print("=" * 80)
print("EXPOSURE & WEIGHTING")
print("=" * 80)


# =============================================================================
# Portfolio exposure
# =============================================================================

portfolio_exposure = (
    inverse_volatility_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .sum()
)

assert np.allclose(
    portfolio_exposure.values,
    1.00,
), (
    "Portfolio exposure is not equal to 1.00."
)

print(
    "✓ Portfolio exposure = 1.00 "
    "for all portfolios."
)


# =============================================================================
# Weight differentiation
# =============================================================================

weight_check = (
    inverse_volatility_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .agg(
        min_weight="min",
        max_weight="max",
    )
)

assert (
    weight_check["max_weight"]
    > weight_check["min_weight"]
).all(), (
    "Inverse Volatility does not produce "
    "differentiated weights."
)

print(
    "✓ Inverse Volatility produces "
    "differentiated weights."
)

EXPOSURE & WEIGHTING
✓ Portfolio exposure = 1.00 for all portfolios.
✓ Inverse Volatility produces differentiated weights.


In [43]:
# =============================================================================
# Validation: Position Count
# =============================================================================

print("=" * 80)
print("POSITION COUNT")
print("=" * 80)


for model in inverse_volatility_weights["model"].unique():

    model_data = (
        inverse_volatility_weights[
            inverse_volatility_weights["model"] == model
        ]
    )

    counts = (
        model_data
        .groupby(
            [
                "date",
                "portfolio",
            ]
        )["ticker"]
        .nunique()
        .unstack("portfolio")
    )

    print(
        f"✓ {model}: "
        f"Top 10% = "
        f"{counts['long_only_top_10_inverse_volatility'].min()}–"
        f"{counts['long_only_top_10_inverse_volatility'].max()} | "
        f"Top 20% = "
        f"{counts['long_only_top_20_inverse_volatility'].min()}–"
        f"{counts['long_only_top_20_inverse_volatility'].max()} | "
        f"Top 30% = "
        f"{counts['long_only_top_30_inverse_volatility'].min()}–"
        f"{counts['long_only_top_30_inverse_volatility'].max()}"
    )

POSITION COUNT
✓ Ridge: Top 10% = 50–50 | Top 20% = 99–100 | Top 30% = 149–149
✓ XGBoost: Top 10% = 50–50 | Top 20% = 99–100 | Top 30% = 149–149
✓ Random Forest: Top 10% = 50–50 | Top 20% = 99–100 | Top 30% = 149–149


In [44]:
# =============================================================================
# Validation: Inverse Volatility Weighting
# =============================================================================

print("=" * 80)
print("INVERSE VOLATILITY WEIGHTING")
print("=" * 80)


max_error = 0.0


for (date, model, portfolio), group in (
    inverse_volatility_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )
):

    tickers = group["ticker"].tolist()

    actual_weights = (
        group["weight"]
        .to_numpy()
    )

    volatilities = (
        rolling_volatility
        .loc[date, tickers]
        .to_numpy()
    )

    expected_weights = (
        1.0 / volatilities
    )

    expected_weights /= (
        expected_weights.sum()
    )

    error = np.max(
        np.abs(
            actual_weights
            - expected_weights
        )
    )

    max_error = max(
        max_error,
        error,
    )


# =============================================================================
# Validation
# =============================================================================

assert np.isclose(
    max_error,
    0.0,
    atol=1e-10,
), (
    "Inverse Volatility weights do not match "
    "the expected inverse-volatility allocation."
)

print(
    "✓ Portfolio weights match the "
    "inverse-volatility allocation."
)

print(
    f"✓ Maximum absolute error = "
    f"{max_error:.2e}"
)

INVERSE VOLATILITY WEIGHTING
✓ Portfolio weights match the inverse-volatility allocation.
✓ Maximum absolute error = 4.16e-17


#### 6.2.2 Risk Parity

    

In [45]:
from src.portfolio.risk_parity import compute_risk_contributions
from src.portfolio.risk_parity import compute_risk_parity_weights


# =============================================================================
# Risk Parity — Portfolio Construction
# =============================================================================

SELECTION_LEVELS = {
    "top_10": 10,
    "top_20": 9,
    "top_30": 8,
}

risk_parity_weights_list = []
risk_parity_covariances = {}


# =============================================================================
# Date → Integer Position Mapping
# =============================================================================

date_to_pos = {
    date: i
    for i, date in enumerate(log_returns.index)
}


# =============================================================================
# Build portfolios
# =============================================================================

for model, quantile_column in QUANTILE_COLUMNS.items():

    for portfolio_name, minimum_quantile in SELECTION_LEVELS.items():

        # ---------------------------------------------------------------------
        # Select assets
        # ---------------------------------------------------------------------

        selected = (
            quantile_data[quantile_column]
            >= minimum_quantile
        )

        selected_index = (
            quantile_data.index[selected]
        )

        # ---------------------------------------------------------------------
        # Process each date independently
        # ---------------------------------------------------------------------

        for date in selected_index.get_level_values(
            "date"
        ).unique():

            date_index = selected_index[
                selected_index.get_level_values("date")
                == date
            ]

            tickers = (
                date_index
                .get_level_values("ticker")
                .tolist()
            )

            # -----------------------------------------------------------------
            # Historical 21-day return window
            # -----------------------------------------------------------------

            pos = date_to_pos[date]

            if pos < VOLATILITY_WINDOW:
                continue

            returns_window = (
                log_returns
                .iloc[
                    pos - VOLATILITY_WINDOW : pos
                ][tickers]
            )

            # -----------------------------------------------------------------
            # Skip incomplete windows
            # -----------------------------------------------------------------

            if len(returns_window) < VOLATILITY_WINDOW:
                continue

            # -----------------------------------------------------------------
            # Remove assets with missing observations
            # -----------------------------------------------------------------

            valid_tickers = (
                returns_window
                .columns[
                    returns_window
                    .notna()
                    .all()
                ]
                .tolist()
            )

            returns_window = (
                returns_window[
                    valid_tickers
                ]
            )

            if len(valid_tickers) < 2:
                continue

            # -----------------------------------------------------------------
            # Risk Parity optimization
            # -----------------------------------------------------------------

            weights, covariance_matrix = (
                compute_risk_parity_weights(
                    returns_window.values
                )
            )

            portfolio_name_full = (
                f"long_only_{portfolio_name}_risk_parity"
            )

            # -----------------------------------------------------------------
            # Store covariance matrix
            # -----------------------------------------------------------------

            risk_parity_covariances[
                (
                    date,
                    model,
                    portfolio_name_full,
                )
            ] = covariance_matrix

            # -----------------------------------------------------------------
            # Build date-level weights
            # -----------------------------------------------------------------

            date_weights = pd.DataFrame(
                {
                    "date": date,
                    "ticker": valid_tickers,
                    "weight": weights,
                }
            )

            date_weights["model"] = model
            date_weights["portfolio"] = portfolio_name_full

            risk_parity_weights_list.append(
                date_weights[
                    [
                        "date",
                        "ticker",
                        "model",
                        "portfolio",
                        "weight",
                    ]
                ]
            )


# =============================================================================
# Combine portfolios
# =============================================================================

risk_parity_weights = pd.concat(
    risk_parity_weights_list,
    ignore_index=True,
)

La validación de Risk Parity se estructura en cuatro bloques para comprobar progresivamente la correcta construcción de las carteras. Se verifica primero la integridad de los pesos, seguida de su exposición y normalización, la coherencia del número de posiciones con la selección por cuantiles y, finalmente, que las posiciones produzcan una contribución al riesgo aproximadamente equilibrada, evaluando además cómo varía esta precisión según el nivel de selección.

In [46]:
# =============================================================================
# Validation: Portfolio Integrity
# =============================================================================

print("=" * 80)
print("PORTFOLIO INTEGRITY")
print("=" * 80)


# =============================================================================
# 1. No negative weights
# =============================================================================

assert (
    risk_parity_weights["weight"] > 0
).all(), (
    "Non-positive weights detected."
)

print("✓ All portfolio weights are strictly positive.")


# =============================================================================
# 2. No duplicated observations
# =============================================================================

duplicates = (
    risk_parity_weights
    .duplicated(
        subset=[
            "date",
            "ticker",
            "model",
            "portfolio",
        ]
    )
    .sum()
)

assert duplicates == 0, (
    f"Found {duplicates} duplicated "
    "date-ticker-model-portfolio observations."
)

print(
    "✓ No duplicated "
    "date-ticker-model-portfolio observations."
)

PORTFOLIO INTEGRITY
✓ All portfolio weights are strictly positive.
✓ No duplicated date-ticker-model-portfolio observations.


In [47]:
# =============================================================================
# Validation: Exposure & Weighting
# =============================================================================

print("=" * 80)
print("EXPOSURE & WEIGHTING")
print("=" * 80)


# =============================================================================
# Portfolio exposure
# =============================================================================

portfolio_exposure = (
    risk_parity_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .sum()
)

assert np.allclose(
    portfolio_exposure.values,
    1.00,
), (
    "Portfolio exposure is not equal to 1.00."
)

print(
    "✓ Portfolio exposure = 1.00 "
    "for all portfolios."
)


# =============================================================================
# Weight differentiation
# =============================================================================

weight_check = (
    risk_parity_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )["weight"]
    .agg(
        min_weight="min",
        max_weight="max",
    )
)

assert (
    weight_check["max_weight"]
    > weight_check["min_weight"]
).all(), (
    "Risk Parity does not produce "
    "differentiated weights."
)

print(
    "✓ Risk Parity produces "
    "differentiated weights."
)

EXPOSURE & WEIGHTING
✓ Portfolio exposure = 1.00 for all portfolios.
✓ Risk Parity produces differentiated weights.


In [48]:
# =============================================================================
# Validation: Position Count
# =============================================================================

print("=" * 80)
print("POSITION COUNT")
print("=" * 80)


for model in risk_parity_weights["model"].unique():

    model_data = (
        risk_parity_weights[
            risk_parity_weights["model"] == model
        ]
    )

    counts = (
        model_data
        .groupby(
            [
                "date",
                "portfolio",
            ]
        )["ticker"]
        .nunique()
        .unstack("portfolio")
    )

    print(
        f"✓ {model}: "
        f"Top 10% = "
        f"{counts['long_only_top_10_risk_parity'].min()}–"
        f"{counts['long_only_top_10_risk_parity'].max()} | "
        f"Top 20% = "
        f"{counts['long_only_top_20_risk_parity'].min()}–"
        f"{counts['long_only_top_20_risk_parity'].max()} | "
        f"Top 30% = "
        f"{counts['long_only_top_30_risk_parity'].min()}–"
        f"{counts['long_only_top_30_risk_parity'].max()}"
    )

POSITION COUNT
✓ Ridge: Top 10% = 50–50 | Top 20% = 99–100 | Top 30% = 149–149
✓ XGBoost: Top 10% = 50–50 | Top 20% = 99–100 | Top 30% = 149–149
✓ Random Forest: Top 10% = 50–50 | Top 20% = 99–100 | Top 30% = 149–149


In [49]:
# =============================================================================
# Validation: Equal Risk Contribution
# =============================================================================

print("=" * 80)
print("EQUAL RISK CONTRIBUTION")
print("=" * 80)


risk_contribution_errors = []


# =============================================================================
# Compute risk contribution deviations
# =============================================================================

for (date, model, portfolio), group in (
    risk_parity_weights
    .groupby(
        [
            "date",
            "model",
            "portfolio",
        ]
    )
):

    weights = (
        group["weight"]
        .to_numpy()
    )

    covariance_matrix = np.asarray(
        risk_parity_covariances[
            (date, model, portfolio)
        ]
    )

    # -------------------------------------------------------------------------
    # Dimension checks
    # -------------------------------------------------------------------------

    assert covariance_matrix.shape == (
        len(weights),
        len(weights),
    ), (
        f"Covariance matrix dimensions do not match "
        f"portfolio size for {date}, {model}, {portfolio}."
    )

    # -------------------------------------------------------------------------
    # Risk contributions
    # -------------------------------------------------------------------------

    risk_contributions = (
        compute_risk_contributions(
            weights,
            covariance_matrix,
        )
    )

    # -------------------------------------------------------------------------
    # Target risk contribution
    # -------------------------------------------------------------------------

    target_risk = (
        risk_contributions.sum()
        / len(risk_contributions)
    )

    # -------------------------------------------------------------------------
    # Relative deviation from target
    # -------------------------------------------------------------------------

    max_relative_error = (
        np.max(
            np.abs(
                risk_contributions
                - target_risk
            )
        )
        / target_risk
    )

    risk_contribution_errors.append(
        {
            "date": date,
            "model": model,
            "portfolio": portfolio,
            "max_relative_error": max_relative_error,
        }
    )


# =============================================================================
# Convert results to DataFrame
# =============================================================================

risk_contribution_errors = pd.DataFrame(
    risk_contribution_errors
)


# =============================================================================
# Overall summary
# =============================================================================

print(
    f"✓ Portfolios validated = "
    f"{len(risk_contribution_errors):,}"
)

print(
    f"✓ Mean relative deviation = "
    f"{risk_contribution_errors['max_relative_error'].mean():.6%}"
)

print(
    f"✓ Median relative deviation = "
    f"{risk_contribution_errors['max_relative_error'].median():.6%}"
)

print(
    f"✓ 95th percentile = "
    f"{risk_contribution_errors['max_relative_error'].quantile(0.95):.6%}"
)

print(
    f"✓ Maximum relative deviation = "
    f"{risk_contribution_errors['max_relative_error'].max():.6%}"
)


# =============================================================================
# Summary by selection level
# =============================================================================

risk_contribution_summary = (
    risk_contribution_errors
    .assign(
        selection=(
            risk_contribution_errors["portfolio"]
            .str.extract(
                r"(top_\d+)"
            )[0]
        )
    )
    .groupby("selection")["max_relative_error"]
    .agg(
        mean="mean",
        median="median",
        p95=lambda x: x.quantile(0.95),
        max="max",
    )
    .reindex(
        [
            "top_10",
            "top_20",
            "top_30",
        ]
    )
)


print()
print("=" * 80)
print("RISK CONTRIBUTION DEVIATION BY SELECTION LEVEL")
print("=" * 80)

print(
    risk_contribution_summary
    .to_string(
        float_format=lambda x: f"{x:.2%}"
    )
)


EQUAL RISK CONTRIBUTION
✓ Portfolios validated = 3,348
✓ Mean relative deviation = 0.277731%
✓ Median relative deviation = 0.209516%
✓ 95th percentile = 0.734043%
✓ Maximum relative deviation = 2.133380%

RISK CONTRIBUTION DEVIATION BY SELECTION LEVEL
           mean  median   p95   max
selection                          
top_10    0.11%   0.10% 0.25% 0.62%
top_20    0.28%   0.24% 0.58% 1.06%
top_30    0.45%   0.39% 0.99% 2.13%


Las desviaciones observadas en la asignación de Risk Parity no constituyen un defecto de implementación, sino una decisión metodológica motivada por el compromiso entre dimensionalidad y ventana temporal de estimación ($21$ sesiones). Al ampliar el universo seleccionado —desde $\sim 50$ activos en el Top 10% hasta $\sim 150$ en el Top 30%—, aumenta la exigencia sobre la matriz de covarianzas, reflejándose en un incremento progresivo de la desviación media (del $0,11\%$ en Top 10% al $0,45\%$ en Top 30%) y de los máximos observados ($0,62\%$ frente a $2,13\%$).Aunque el uso de Ledoit-Wolf Shrinkage ya regulariza adecuadamente la matriz, forzar tolerancias más estrictas en los universos ampliados solo ajustaría el solver al ruido de estimación. Puesto que el universo principal de inversión es el Top 10% —donde la precisión es prácticamente perfecta (mediana de $0,096\%$ y máximo de $0,62\%$ )—, la precisión alcanzada se considera óptima y suficiente para los objetivos del estudio.


### 6.3 Mathematical Optimization (Mean-Variance / Maximum Sharpe)

En esta sección se introduce una metodología de construcción de carteras que combina directamente la señal predictiva del modelo con la estructura de riesgo de los activos seleccionados. A diferencia de los métodos anteriores, donde la predicción se utilizaba principalmente para seleccionar activos o determinar la intensidad de sus pesos, aquí la señal se transforma en una estimación de retorno activo esperado y se incorpora explícitamente al problema de optimización.

El procedimiento se divide en dos etapas. En primer lugar, la posición relativa proporcionada por el modelo se calibra para obtener una estimación de alpha esperado. En segundo lugar, esta expectativa de alpha se combina con una matriz de covarianzas de los activos mediante un problema de optimización media-varianza para determinar los pesos de la cartera.

Esta separación permite distinguir dos fuentes de valor: la calidad de la señal predictiva, responsable de identificar oportunidades de inversión, y la calidad de la construcción de cartera, responsable de asignar el capital de forma eficiente teniendo en cuenta simultáneamente rentabilidad esperada y riesgo. De este modo, se puede evaluar si una mejora en el comportamiento de las carteras procede de una mejor selección de activos o de una asignación más eficiente del capital.


#### 6.3.1 Signal Calibration — Rank to Expected Alpha

Los modelos de Machine Learning utilizados en este trabajo se entrenan para optimizar pérdidas continuas y seleccionar los mejores activos. Sin embargo, debido a la compresión de varianza (shrinkage) típica de los árboles de decisión en entornos con baja relación señal-ruido, la escala absoluta de las predicciones brutas ($\texttt{pred}$) no refleja la magnitud real de los retornos futuros ($\texttt{target}$). No obstante, su orden relativo (cross-sectional ranking) conserva una elevada capacidad predictiva. Por este motivo, la señal no se utiliza para estimar rentabilidades absolutas directas, sino para determinar el orden de preferencia de los activos, aplicando posteriormente una etapa de calibración que transforma la posición relativa de cada activo en una estimación de alpha esperado:

$$r_{i,t}^{\text{rank}} \longrightarrow \hat{\alpha}_{i,t}$$

**Diseño Metodológico de la Calibración**

Para construir la función de calibración de forma transparente y sin sesgos de información (data leakage), se implementa la siguiente cadena de procesamiento:

1. **Predicciones Out-Of-Fold (OOF) y Promediado de Caminos (Path Averaging)**: Las predicciones históricas proceden exclusivamente de la fase de Combinatorial Purged Cross-Validation (CPCV), garantizando que el modelo jamás vio esas observaciones durante su entrenamiento. Dado que el esquema CPCV evalúa una misma fecha en múltiples combinaciones ($\binom{N}{k}$ paths), las predicciones repetidas para cada pareja $(\text{date}, \text{ticker})$ se promedian en un valor único $\bar{p}_{i,t}$. Este ensemble de caminos reduce la varianza de la estimación y asegura la unicidad muestral.

2. **Normalización por Percentil Cross-Sectional**: En cada fecha $t$, el valor $\bar{p}_{i,t}$ se convierte en su percentil relativo $\text{pct}_{i,t} = \text{rank}(\bar{p}_{i,t}) / N_t \in [0, 1]$. Esta transformación aísla la señal de posibles desplazamientos en la escala global del modelo a lo largo del tiempo.

3. **Purga Temporal de 21 Días**: Para ajustar la curva en una fecha $t$, solo se utilizan parejas históricas $(\text{pct}_{i,t'}, r_{i,t':t'+21})$ cuya ventana de retorno a 21 días ya haya finalizado por completo ($t' + 21 \le t$). Esto evita el look-ahead bias por la vía del target.

4. **Discretización en Deciles y Regresión Isotónica**: Las parejas históricas válidas se agrupan en 10 deciles de percentil, calculando el retorno medio realizado en cada uno. Sobre estos puntos discretizados se ajusta una Regresión Isotónica (algoritmo Pool Adjacent Violators), imponiendo una restricción monótona no decreciente que evita el sobreajuste a observaciones individuales ruidosas.

5. **Ventana Expanding y Frecuencia Mensual**: La calibración aprovecha todo el histórico purgado disponible hasta $t$ (expanding window), maximizando el tamaño muestral por decil. El reajuste se ejecuta con frecuencia mensual para evitar reaccionar al ruido de corto plazo.


**Sesgo en el Arranque Inicial y Disolución Progresiva**

Al inicio del periodo de evaluación Out-Of-Sample (OOS), no existen retornos reales $r_{21d}$ finalizados dentro de la ventana de test. Por tanto, la tabla de calibración inicial se construye sobre las predicciones OOF de CPCV del periodo de train/validation.

Esto introduce un sesgo de arranque menor y de signo opuesto al sobreajuste convencional: al proceder de submodelos CPCV entrenados con una fracción de datos $(K-1)/K$, la capacidad discriminativa inicial es ligeramente inferior a la del modelo final en producción. A medida que avanza el periodo OOS, las nuevas parejas $(\text{pct}, r_{21d})$ reales generadas por el modelo definitivo se incorporan a la ventana expanding, diluyendo este sesgo inicial hasta volverlo marginal transcurridos unos meses. Como control de robustez, se evalúa la discrepancia entre la curva basada en CPCV y la curva ajustada exclusivamente con parejas reales cuando la muestra OOS es suficiente.


In [ ]:
from sklearn.isotonic import IsotonicRegression

oof_predictions_xgb = pd.read_parquet("../data/model_results/oof_predictions/oof_preds_xgb_rank.parquet")
oof_predictions_rf = pd.read_parquet("../data/model_results/oof_predictions/oof_preds_rf_rank.parquet")
oof_predictions_rige = pd.read_parquet("../data/model_results/oof_predictions/oof_preds_ridge_rank.parquet")


# =============================================================================
# 6.3.1 — Signal Calibration
# =============================================================================

OOF_PATHS = {
    "Ridge": oof_predictions_rige,
    "XGBoost": oof_predictions_xgb,
    "Random Forest": oof_predictions_rf,
}

TARGET_COLUMN = "target"
PREDICTION_COLUMN = "pred"

CALIBRATION_BINS = 10
CALIBRATION_HORIZON = 21
CALIBRATION_FREQUENCY = "MS"

# =============================================================================
# Build path-averaged OOF predictions
# =============================================================================

oof_calibration_data = {}


for model, data in OOF_PATHS.items():

    data = data.copy()

    # -------------------------------------------------------------------------
    # Verify that target is unique for each date-ticker pair
    # -------------------------------------------------------------------------

    target_counts = (
        data[TARGET_COLUMN]
        .groupby(level=["date", "ticker"])
        .nunique()
    )

    assert (
        target_counts.max() == 1
    ), (
        f"{model}: target is not unique "   
        "for some date-ticker observations."
    )

    # -------------------------------------------------------------------------
    # Average predictions across CPCV paths
    # -------------------------------------------------------------------------

    calibration = (
        data
        .groupby(level=["date", "ticker"])
        .agg(
            pred=(
                PREDICTION_COLUMN,
                "mean",
            ),
            target=(
                TARGET_COLUMN,
                "first",
            ),
            n_paths=(
                PREDICTION_COLUMN,
                "count",
            ),
        )
    )

    oof_calibration_data[model] = calibration


# =============================================================================
# OOF calibration sample
# =============================================================================

for model, data in oof_calibration_data.items():

    print("=" * 80)
    print(model)
    print("=" * 80)

    print(
        f"Unique observations: {len(data):,}"
    )

    print(
        f"Date range: "
        f"{data.index.get_level_values('date').min().date()} "
        f"→ "
        f"{data.index.get_level_values('date').max().date()}"
    )

    print(
        f"Average CPCV paths: "
        f"{data['n_paths'].mean():.2f}"
    )

    print()

Ridge
Unique observations: 1,627,370
Date range: 2011-01-03 → 2024-11-27
Average CPCV paths: 6.00

XGBoost
Unique observations: 1,627,370
Date range: 2011-01-03 → 2024-11-27
Average CPCV paths: 6.00

Random Forest
Unique observations: 1,627,370
Date range: 2011-01-03 → 2024-11-27
Average CPCV paths: 6.00



In [51]:
# =============================================================================
# Cross-sectional percentile and deciles
# =============================================================================

for model, data in oof_calibration_data.items():

    # -------------------------------------------------------------------------
    # Cross-sectional percentile
    # -------------------------------------------------------------------------

    data["percentile"] = (
        data["pred"]
        .groupby(level="date")
        .rank(
            method="first",
            pct=True,
        )
    )

    # -------------------------------------------------------------------------
    # Decile
    # -------------------------------------------------------------------------

    data["decile"] = (
        np.ceil(
            data["percentile"]
            * CALIBRATION_BINS
        )
        .clip(
            upper=CALIBRATION_BINS
        )
        .astype(int)
    )

In [55]:


# =============================================================================
# Raw calibration curve
# =============================================================================

for model, data in oof_calibration_data.items():

    calibration_curve = (
        data
        .groupby("decile")["target"]
        .agg(
            mean_return="mean",
            observations="count",
        )
    )

    print("=" * 80)
    print(f"{model} — OOF CALIBRATION CURVE")
    print("=" * 80)

    print(
        calibration_curve.to_string(
            float_format=lambda x: f"{x:.6f}"
        )
    )

    print()

Ridge — OOF CALIBRATION CURVE
        mean_return  observations
decile                           
1          0.010942        161388
2          0.012046        162578
3          0.010988        162678
4          0.012040        162757
5          0.013286        163503
6          0.013163        162159
7          0.015264        162440
8          0.016159        162995
9          0.017373        162261
10         0.026423        164611

XGBoost — OOF CALIBRATION CURVE
        mean_return  observations
decile                           
1          0.013040        161388
2          0.011069        162578
3          0.012594        162678
4          0.010809        162757
5          0.011433        163503
6          0.013305        162159
7          0.013881        162440
8          0.016846        162995
9          0.017873        162261
10         0.026853        164611

Random Forest — OOF CALIBRATION CURVE
        mean_return  observations
decile                           
1          0.0

Los resultados de la curva de calibración cruda muestran una clara asimetría en la capacidad predictiva de las tres arquitecturas evaluadas: mientras que los modelos destacan discriminando los activos ganadores en la cola superior, pierden resolución al ordenar la parte baja de la tabla.

En la parte alta (deciles 7 a 10), la ordenación es limpia y estrictamente creciente. El retorno medio a 21 días pasa progresivamente del ~1.50% hasta alcanzar su máximo en el Decil 10, donde se sitúa en torno al 2.65% en todos los algoritmos. Este comportamiento confirma que las señales basadas en factores identifican con precisión las dinámicas operativas y de momento que impulsan a las mejores empresas del mercado.

Por el contrario, la cola inferior (deciles 1 a 4) presenta un perfil plano y ruidoso. Los rendimientos de estos grupos quedan estancados en una franja estrecha entre el 1.08% y el 1.30%, registrando incluso pequeñas inversiones de orden donde el Decil 1 rinde ligeramente más que el Decil 4 en XGBoost y Random Forest. Esta limitación refleja que el deterioro de los activos de menor calidad responde a eventos más impredecibles y difíciles de ordenar linealmente mediante el conjunto de factores utilizado.

Esta conducta justifica de forma directa la aplicación de la Regresión Isotónica. El algoritmo corregirá la falta de monotonía en la cola inferior fusionando los deciles 1 a 4 en un único escalón horizontal de retornos esperados ($\hat{\alpha} \approx 1.15\%$). De este modo, se evita asignar valoraciones engañosas en tramos donde el modelo no demuestra capacidad discriminativa, preservando al mismo tiempo la gradación creciente en los deciles superiores.

In [53]:
# =============================================================================
# Isotonic calibration
# =============================================================================

def fit_isotonic_calibration(
    calibration_data,
):
    """
    Fit isotonic regression from cross-sectional
    prediction percentile to realized forward return.
    """

    # -------------------------------------------------------------------------
    # Aggregate realized returns by decile
    # -------------------------------------------------------------------------

    decile_data = (
        calibration_data
        .groupby("decile")
        .agg(
            percentile=("percentile", "mean"),
            target=("target", "mean"),
            observations=("target", "count"),
        )
        .reset_index()
    )

    # -------------------------------------------------------------------------
    # Fit isotonic regression
    # -------------------------------------------------------------------------

    isotonic_model = IsotonicRegression(
        increasing=True,
        out_of_bounds="clip",
    )

    isotonic_model.fit(
        decile_data["percentile"],
        decile_data["target"],
        sample_weight=decile_data["observations"],
    )

    return (
        isotonic_model,
        decile_data,
    )


In [54]:

# =============================================================================
# Fit initial OOF calibration
# =============================================================================

oof_calibrators = {}
oof_calibration_curves = {}


for model, data in oof_calibration_data.items():

    calibrator, curve = (
        fit_isotonic_calibration(
            data
        )
    )

    oof_calibrators[model] = calibrator
    oof_calibration_curves[model] = curve

In [57]:
# =============================================================================
# OOF Isotonic Calibration Curve
# =============================================================================

for model, data in oof_calibration_data.items():

    calibrator = oof_calibrators[model]

    calibration_curve = (
        data
        .groupby("decile")
        .agg(
            percentile=("percentile", "mean"),
            mean_return=("target", "mean"),
            observations=("target", "count"),
        )
    )

    # -------------------------------------------------------------------------
    # Isotonic calibrated expected return
    # -------------------------------------------------------------------------

    calibration_curve["calibrated_return"] = (
        calibrator.predict(
            calibration_curve["percentile"]
        )
    )

    print("=" * 80)
    print(f"{model} — OOF ISOTONIC CALIBRATION")
    print("=" * 80)

    print(
        calibration_curve[
            [
                "percentile",
                "mean_return",
                "calibrated_return",
                "observations",
            ]
        ].to_string(
            float_format=lambda x: f"{x:.6f}"
        )
    )

    print()

Ridge — OOF ISOTONIC CALIBRATION
        percentile  mean_return  calibrated_return  observations
decile                                                          
1         0.050662     0.010942           0.010942        161388
2         0.150198     0.012046           0.011517        162578
3         0.250131     0.010988           0.011517        162678
4         0.350119     0.012040           0.012040        162757
5         0.450360     0.013286           0.013225        163503
6         0.550418     0.013163           0.013225        162159
7         0.650149     0.015264           0.015264        162440
8         0.750137     0.016159           0.016159        162995
9         0.850070     0.017373           0.017373        162261
10        0.950498     0.026423           0.026423        164611

XGBoost — OOF ISOTONIC CALIBRATION
        percentile  mean_return  calibrated_return  observations
decile                                                          
1         0.050662   

Tras ajustar la Regresión Isotónica sobre las curvas crudas, se observa cómo el algoritmo corrige con éxito las inconsistencias de ordenación, generando una función de retorno esperado (`calibrated_return`) estrictamente creciente para los tres modelos.

En la cola inferior, el algoritmo resuelve la falta de monotonía aplanando los tramos ruidosos. En XGBoost y Random Forest, donde los rendimientos medios reales (`mean_return`) presentaban fluctuaciones sin orden claro entre los deciles 1 y 4 (e incluso el decil 5 en XGBoost), la Regresión Isotónica colapsa estos grupos en una meseta horizontal con un alpha esperado uniforme ($\approx 1.16\% - 1.18\%$). En Ridge, este aplanamiento ocurre de forma más localizada entre los deciles 2 y 3 ($\approx 1.15\%$). De este modo, se elimina el ruido en las zonas donde la señal no demuestra capacidad discriminativa real, evitando asignar penalizaciones o bonificaciones artificiales.

Por el contrario, en la cola superior el ajuste isotónico respeta la gradación natural de la señal. A partir del decil 6 o 7, donde la relación con la rentabilidad es monótona por naturaleza, la curva calibrada mantiene intactos los escalones crecientes observados hasta alcanzar su máximo en el decil 10, asegurando que los activos con mayor fuerza relativa reciban su correspondiente bonificación de alpha ($\approx 2.48\% - 2.68\%$).

Con este bloque hemos construido y verificado la relación histórica entre la posición relativa de la señal y el retorno futuro, utilizando exclusivamente las predicciones *Out-Of-Fold* (OOF) procedentes del CPCV. El resultado proporciona una primera curva de calibración percentil $\to$ retorno esperado, pero todavía no constituye la calibración definitiva utilizada durante el *backtest Out-Of-Sample* (OOS).

A continuación, trasladamos esta metodología al periodo OOS mediante una ventana *expanding* y una recalibración de frecuencia mensual. Para cada fecha de inversión, la función de calibración utilizará únicamente la información histórica cuyo *target* a 21 días ya haya sido observado, aplicando así la purga temporal necesaria para evitar sesgos de anticipación (*look-ahead bias*). De esta forma se obtiene, para cada activo en el periodo OOS, una estimación de alpha esperado ($\hat{\alpha}$) limpia y matemáticamente sólida, lista para alimentar el optimizador de cartera.

In [62]:
# =============================================================================
# OOS Signal Percentiles
# =============================================================================

OOS_SIGNAL_COLUMNS = {
    "Ridge": "prediction_ridge_rank",
    "XGBoost": "prediction_xgb_rank",
    "Random Forest": "prediction_rf_rank",
}

oos_calibration_data = oos_predictions.copy()


for model, signal_column in OOS_SIGNAL_COLUMNS.items():

    percentile_column = (
        f"{signal_column}_percentile"
    )

    # -------------------------------------------------------------------------
    # Cross-sectional percentile
    # -------------------------------------------------------------------------

    oos_calibration_data[
        percentile_column
    ] = (
        oos_calibration_data[
            signal_column
        ]
        .groupby(level="date")
        .rank(
            method="first",
            pct=True,
        )
    )

In [63]:
# =============================================================================
# Validation: OOS Signal Percentiles
# =============================================================================

for model, signal_column in OOS_SIGNAL_COLUMNS.items():

    percentile_column = (
        f"{signal_column}_percentile"
    )

    print("=" * 80)
    print(f"{model} — OOS SIGNAL PERCENTILE")
    print("=" * 80)

    print(
        oos_calibration_data[
            percentile_column
        ]
        .describe()
        .to_string()
    )

    print()

Ridge — OOS SIGNAL PERCENTILE
count    184408.000000
mean          0.501009
std           0.288675
min           0.002016
25%           0.251012
50%           0.501010
75%           0.751012
max           1.000000

XGBoost — OOS SIGNAL PERCENTILE
count    184408.000000
mean          0.501009
std           0.288675
min           0.002016
25%           0.251012
50%           0.501010
75%           0.751012
max           1.000000

Random Forest — OOS SIGNAL PERCENTILE
count    184408.000000
mean          0.501009
std           0.288675
min           0.002016
25%           0.251012
50%           0.501010
75%           0.751012
max           1.000000



Los estadísticos de la señal en el periodo *Out-Of-Sample* (OOS) confirman una distribución uniforme estándar idéntica y precisa en los tres modelos sobre las 184.408 observaciones.

Con una media y mediana centradas en $0.501$, y una desviación estándar de $0.2887$ (que coincide con el valor teórico de una uniforme, $1/\sqrt{12}$), los cuartiles ($0.251$, $0.501$, $0.751$) y el rango $[0.002, 1.000]$ reflejan la perfecta estandarización de las predicciones en la escala $[0, 1]$.

Este comportamiento garantiza que la transformación *cross-sectional* diaria elimina cualquier sesgo de magnitud o escala entre modelos. Las entradas que recibe el calibrador dinámico en el periodo OOS son homogéneas y estables en todo el horizonte temporal, asegurando la asignación limpia del alpha esperado ($\hat{\alpha}$) para la optimización de cartera.

In [71]:
# =============================================================================
# OOF Signal Percentiles
# =============================================================================

for model, data in oof_calibration_data.items():

    data["percentile"] = (
        data["pred"]
        .groupby(level="date")
        .rank(
            method="first",
            pct=True,
        )
    )

# =============================================================================
# Validation: Cross-Sectional Percentiles
# =============================================================================

print("=" * 80)
print("CROSS-SECTIONAL PERCENTILE VALIDATION")
print("=" * 80)

for model, data in oof_calibration_data.items():

    percentiles = data["percentile"]

    assert percentiles.min() >= 0
    assert percentiles.max() <= 1

    print(
        f"✓ {model}: "
        f"Range = [{percentiles.min():.6f}, "
        f"{percentiles.max():.6f}] | "
        f"Median = {percentiles.median():.6f}"
    )

CROSS-SECTIONAL PERCENTILE VALIDATION
✓ Ridge: Range = [0.002020, 1.000000] | Median = 0.501085
✓ XGBoost: Range = [0.002020, 1.000000] | Median = 0.501085
✓ Random Forest: Range = [0.002020, 1.000000] | Median = 0.501085


In [73]:
from sklearn.isotonic import IsotonicRegression

# =============================================================================
# OOF Calibration Curves
# =============================================================================

oof_calibrators = {}
oof_calibration_curves = {}


for model, data in oof_calibration_data.items():

    calibration_data = data.copy()

    # -------------------------------------------------------------------------
    # Assign cross-sectional deciles
    # -------------------------------------------------------------------------

    calibration_data["decile"] = (
        pd.qcut(
            calibration_data["percentile"],
            q=10,
            labels=False,
            duplicates="drop",
        )
        + 1
    )

    # -------------------------------------------------------------------------
    # Compute mean realized return by decile
    # -------------------------------------------------------------------------

    curve = (
        calibration_data
        .groupby("decile")["target"]
        .agg(
            mean_return="mean",
            observations="count",
        )
        .reset_index()
    )

    # -------------------------------------------------------------------------
    # Fit isotonic regression
    # -------------------------------------------------------------------------

    isotonic = IsotonicRegression(
        increasing=True,
        out_of_bounds="clip",
    )

    isotonic.fit(
        curve["decile"],
        curve["mean_return"],
        sample_weight=curve["observations"],
    )

    curve["calibrated_return"] = isotonic.predict(
        curve["decile"]
    )

    # -------------------------------------------------------------------------
    # Store results
    # -------------------------------------------------------------------------

    oof_calibrators[model] = isotonic
    oof_calibration_curves[model] = curve


# =============================================================================
# Validation: OOF Calibration Curves
# =============================================================================

for model, curve in oof_calibration_curves.items():

    print("=" * 80)
    print(f"{model} — OOF CALIBRATION CURVE")
    print("=" * 80)

    print(
        curve.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}",
        )
    )

    print()

Ridge — OOF CALIBRATION CURVE
 decile  mean_return  observations  calibrated_return
      1     0.010986        162818           0.010986
      2     0.012038        162846           0.011465
      3     0.010891        162589           0.011465
      4     0.012122        162697           0.012122
      5     0.013273        162742           0.013229
      6     0.013186        162822           0.013229
      7     0.015237        162657           0.015237
      8     0.016293        162764           0.016293
      9     0.017342        162751           0.017342
     10     0.026482        162684           0.026482

XGBoost — OOF CALIBRATION CURVE
 decile  mean_return  observations  calibrated_return
      1     0.013017        162818           0.011772
      2     0.011070        162846           0.011772
      3     0.012583        162589           0.011772
      4     0.010881        162697           0.011772
      5     0.011310        162742           0.011772
      6     0.01340

La tabla de calibración *Out-Of-Fold* (OOF) confirma el comportamiento esperado tras aplicar la Regresión Isotónica sobre las predicciones de los tres modelos.

En la parte alta de la tabla (deciles 7 a 10), la señal mantiene una ordenación monótona perfecta, alcanzando rentabilidades esperadas de entre el 2.48% y el 2.69% en el Decil 10. Por contra, en los deciles inferiores el algoritmo resuelve las inversiones de orden colapsando los tramos ruidosos en escalones planos de alpha esperado ($\approx 1.16\% - 1.18\%$), evitando asignar penalizaciones artificiales donde el modelo no demuestra capacidad discriminativa.

Con estos resultados queda validada la estructura inicial de la señal. El siguiente paso es implementar la **calibración dinámica mediante ventana *expanding* en el periodo *Out-Of-Sample* (OOS)**. La curva OOF obtenida actuará como la calibración base a fecha 15/01/2025; a partir de ese punto, el sistema recalibrará periódicamente la función incorporando solo las observaciones OOS cuyos retornos a 21 días ya hayan finalizado. Esta purga temporal estricta previene cualquier sesgo de anticipación (*look-ahead bias*) y proporciona una estimación de alpha ($\hat{\alpha}$) realista para la optimización de cartera.

In [76]:
# =============================================================================
# Expanding Calibration — OOS
# =============================================================================

oos_calibration_data = {}

PREDICTION_COLUMNS = {
    "Ridge": "prediction_ridge_rank",
    "XGBoost": "prediction_xgb_rank",
    "Random Forest": "prediction_rf_rank",
}


for model, prediction_column in PREDICTION_COLUMNS.items():

    calibration_data = oos_predictions[
        [
            prediction_column,
            "forward_return_21d",
        ]
    ].copy()

    # -------------------------------------------------------------------------
    # Cross-sectional percentile
    # -------------------------------------------------------------------------

    calibration_data["percentile"] = (
        calibration_data[prediction_column]
        .groupby(level="date")
        .rank(
            method="first",
            pct=True,
        )
    )

    oos_calibration_data[model] = calibration_data

In [77]:
# =============================================================================
# Validation: OOS Calibration Data
# =============================================================================

for model, data in oos_calibration_data.items():

    print("=" * 80)
    print(f"{model} — OOS CALIBRATION DATA")
    print("=" * 80)

    print(
        f"Observations: {len(data):,}"
    )

    print(
        f"Date range: "
        f"{data.index.get_level_values('date').min().date()} "
        f"→ "
        f"{data.index.get_level_values('date').max().date()}"
    )

    print(
        f"Missing percentiles: "
        f"{data['percentile'].isna().sum():,}"
    )

    print()

Ridge — OOS CALIBRATION DATA
Observations: 184,408
Date range: 2025-01-15 → 2026-07-10
Missing percentiles: 0

XGBoost — OOS CALIBRATION DATA
Observations: 184,408
Date range: 2025-01-15 → 2026-07-10
Missing percentiles: 0

Random Forest — OOS CALIBRATION DATA
Observations: 184,408
Date range: 2025-01-15 → 2026-07-10
Missing percentiles: 0



Una vez validada la estabilidad de la señal en el periodo *Out-Of-Sample* (OOS), implementamos el esquema de **recalibración mensual mediante ventana *expanding* con purga temporal**.

En cada fecha de rebalanceo, el modelo de Regresión Isotónica se reajustará combinando el histórico *Out-Of-Fold* (OOF) inicial con las nuevas observaciones OOS cuyo retorno a 21 días ya haya finalizado por completo. Al descartar las posiciones en curso, eliminamos estricta y sistemáticamente cualquier riesgo de *look-ahead bias*.

Este proceso adaptativo actualizará dinámicamente la función percentil $\to$ retorno esperado, asignando a cada activo una estimación de alpha ($\hat{\alpha}$) limpia, monótona y lista para alimentar el optimizador de cartera en la **Sección 6.3.2**.



In [79]:
from sklearn.isotonic import IsotonicRegression

# =============================================================================
# Expanding Calibration — Configurable Recalibration Frequency
# =============================================================================

CALIBRATION_FREQUENCY = "MS"

oos_calibrators = {
    model: {}
    for model in PREDICTION_COLUMNS
}

# =============================================================================
# OOS dates
# =============================================================================

oos_dates = (
    oos_predictions.index
    .get_level_values("date")
    .unique()
    .sort_values()
)

# =============================================================================
# Recalibration dates
# =============================================================================

if CALIBRATION_FREQUENCY == "W":

    recalibration_dates = (
        pd.Series(oos_dates)
        .groupby(
            pd.Series(oos_dates).dt.to_period("W-FRI")
        )
        .first()
        .tolist()
    )

elif CALIBRATION_FREQUENCY == "MS":

    recalibration_dates = (
        pd.Series(oos_dates)
        .groupby(
            pd.Series(oos_dates).dt.to_period("M")
        )
        .first()
        .tolist()
    )

elif CALIBRATION_FREQUENCY == "QS":

    recalibration_dates = (
        pd.Series(oos_dates)
        .groupby(
            pd.Series(oos_dates).dt.to_period("Q")
        )
        .first()
        .tolist()
    )

else:

    raise ValueError(
        "Unsupported CALIBRATION_FREQUENCY. "
        "Use 'W', 'MS', or 'QS'."
    )


# =============================================================================
# Build expanding calibrators
# =============================================================================

for model in PREDICTION_COLUMNS:

    # -------------------------------------------------------------------------
    # OOF historical calibration sample
    # -------------------------------------------------------------------------

    oof_data = (
        oof_calibration_data[model]
        [
            [
                "percentile",
                "target",
            ]
        ]
        .copy()
    )

    # -------------------------------------------------------------------------
    # OOS calibration data
    # -------------------------------------------------------------------------

    oos_data = (
        oos_calibration_data[model]
        .rename(
            columns={
                "forward_return_21d": "target"
            }
        )
        [
            [
                "percentile",
                "target",
            ]
        ]
        .copy()
    )

    # -------------------------------------------------------------------------
    # Recalibrate at selected frequency
    # -------------------------------------------------------------------------

    for calibration_date in recalibration_dates:

        # ---------------------------------------------------------------------
        # OOS observations whose 21-day target has fully matured
        # ---------------------------------------------------------------------

        oos_dates_available = (
            oos_data
            .index
            .get_level_values("date")
            + pd.offsets.BDay(21)
            <= calibration_date
        )

        available_oos = (
            oos_data.loc[oos_dates_available]
        )

        # ---------------------------------------------------------------------
        # Combine OOF + matured OOS observations
        # ---------------------------------------------------------------------

        expanding_data = pd.concat(
            [
                oof_data,
                available_oos,
            ]
        )

        # ---------------------------------------------------------------------
        # Remove missing observations
        # ---------------------------------------------------------------------

        expanding_data = (
            expanding_data
            .dropna(
                subset=[
                    "percentile",
                    "target",
                ]
            )
        )

        if len(expanding_data) == 0:
            continue

        # ---------------------------------------------------------------------
        # Assign deciles
        # ---------------------------------------------------------------------

        expanding_data["decile"] = (
            pd.qcut(
                expanding_data["percentile"],
                q=10,
                labels=False,
                duplicates="drop",
            )
            + 1
        )

        # ---------------------------------------------------------------------
        # Mean realized return by decile
        # ---------------------------------------------------------------------

        curve = (
            expanding_data
            .groupby("decile")["target"]
            .agg(
                mean_return="mean",
                observations="count",
            )
            .reset_index()
        )

        # ---------------------------------------------------------------------
        # Isotonic calibration
        # ---------------------------------------------------------------------

        calibrator = IsotonicRegression(
            increasing=True,
            out_of_bounds="clip",
        )

        calibrator.fit(
            curve["decile"],
            curve["mean_return"],
            sample_weight=curve["observations"],
        )

        # ---------------------------------------------------------------------
        # Store calibrator
        # ---------------------------------------------------------------------

        oos_calibrators[model][
            calibration_date
        ] = calibrator

La frecuencia de recalibración de la función de alpha no se fija de forma arbitraria, sino que se plantea como un **hiperparámetro configurable (`CALIBRATION_FREQUENCY`)**.

Para la implementación inicial adoptamos una **frecuencia mensual**, lo que permite alinear las actualizaciones de la función isotónica con el horizonte del *target* a 21 días y la cadencia de rebalanceo de la cartera.

No obstante, la decisión final se someterá a un **análisis de sensibilidad posterior mediante *walk-forward***. Evaluaremos distintas frecuencias de actualización (semanal, mensual y trimestral) manteniendo la ventana *expanding* y la purga de 21 días, seleccionando la opción que maximice la robustez de las métricas OOS y logre el mejor equilibrio entre estabilidad y adaptabilidad al mercado.

In [80]:
# =============================================================================
# Validation: Expanding Calibration
# =============================================================================

print("=" * 80)
print("EXPANDING CALIBRATION")
print("=" * 80)

for model, calibrators in oos_calibrators.items():

    dates = sorted(calibrators.keys())

    print(f"\n{model}")

    print(
        f"✓ Calibrations generated: {len(dates)}"
    )

    print(
        f"✓ First calibration: "
        f"{dates[0].date()}"
    )

    print(
        f"✓ Last calibration: "
        f"{dates[-1].date()}"
    )

print()
print(
    f"✓ Calibration frequency: "
    f"{CALIBRATION_FREQUENCY}"
)

EXPANDING CALIBRATION

Ridge
✓ Calibrations generated: 19
✓ First calibration: 2025-01-15
✓ Last calibration: 2026-07-01

XGBoost
✓ Calibrations generated: 19
✓ First calibration: 2025-01-15
✓ Last calibration: 2026-07-01

Random Forest
✓ Calibrations generated: 19
✓ First calibration: 2025-01-15
✓ Last calibration: 2026-07-01

✓ Calibration frequency: MS


#### 6.3.2 Portfolio Construction — Mean-Variance Optimization

Una vez obtenidas las estimaciones de alpha esperado ($\hat{\boldsymbol{\alpha}}_t$), estas se combinan con la estructura de dependencia entre los activos seleccionados para determinar los pesos de la cartera mediante optimización media-varianza. El objetivo es asignar el capital buscando maximizar el retorno activo esperado por unidad de riesgo:

$$\max_{\mathbf{w}} \frac{\mathbf{w}^\top \hat{\boldsymbol{\alpha}}_t}{\sqrt{\mathbf{w}^\top \boldsymbol{\Sigma}_t \mathbf{w}}}$$

donde $\hat{\boldsymbol{\alpha}}_t$ representa el vector de alphas calibrados y $\boldsymbol{\Sigma}_t$ la matriz de covarianzas estimada en la fecha $t$.

La estimación del riesgo mantiene la misma metodología utilizada en Risk Parity, empleando una ventana rolling de 21 sesiones de trading con información estrictamente anterior a la fecha de inversión. Dado que el número de activos elegibles supera ampliamente el horizonte temporal de 21 días, la matriz de covarianzas se regulariza mediante Ledoit-Wolf Shrinkage, reduciendo la inestabilidad muestral.

La optimización incorpora restricciones sobre los pesos para evitar concentraciones excesivas. De este modo, el peso de un activo no depende únicamente de la magnitud de su alpha, sino de su interacción conjunta con el resto de la cartera. Un activo con un alpha elevado puede recibir una ponderación reducida si su inclusión incrementa desproporcionadamente el riesgo total por volatilidad o correlación.

Esta aproximación diferencia este método de los anteriores: mientras que Signal Weighting atiende solo a la intensidad de la predicción, Inverse Volatility considera únicamente el riesgo individual y Risk Parity busca igualar la contribución al riesgo, la optimización media-varianza pondera de forma simultánea la expectativa de retorno calibrada y el riesgo conjunto, garantizando una asignación de capital eficiente.


## 7. Portfolio Constraints
Introducimos restricciones de cartera para controlar la concentración, las exposiciones y los riesgos derivados de los distintos métodos de asignación de pesos.

### 7.1 Maximum Position Weight
Limitación del peso máximo que puede alcanzar individualmente cada activo para evitar una concentración excesiva.

### 7.2 Exposure Constraints
Control de las exposiciones long, short, neta y bruta para garantizar que las carteras mantienen el perfil de riesgo definido.

### 7.3 Concentration Control
Evaluación y control de la concentración de las carteras para evitar que los pesos se concentren excesivamente en un reducido número de activos.

### 7.4 Optimization Constraints
Aplicación de restricciones específicas a los métodos de optimización matemática para evitar soluciones extremas, inestables o poco realistas desde el punto de vista de implementación.

### 7.5 Constraint Validation
Verificación de que los pesos finales cumplen todas las restricciones definidas después de completar la construcción de cada cartera.

## 8. Rebalancing
Transformamos las carteras estáticas en estrategias dinámicas mediante la actualización periódica de las posiciones.

### 8.1 Rebalancing Frequency
Definición de la frecuencia de actualización de las carteras, sin asumir automáticamente que el horizonte de predicción de 21 días implica un rebalanceo cada 21 sesiones.

### 8.2 Position Updates & Buffer Rules
Actualización de las posiciones y pesos de acuerdo con las nuevas señales disponibles en cada fecha de rebalanceo, incorporando bandas de tolerancia o reglas de buffer para mitigar ejecuciones marginales innecesarias.

### 8.3 Portfolio Turnover
Cálculo del volumen de posiciones que debe modificarse en cada rebalanceo como medida de la intensidad operativa de la estrategia.

## 9. Transaction Costs & Net Returns
Incorporamos los costes derivados de la implementación de las estrategias para obtener retornos económicamente más realistas.

### 9.1 Transaction Cost & Market Impact Model
Definición de un modelo explícito de costes de transacción que contemple comisiones de ejecución y estimaciones de bid-ask spread e impacto en el mercado aplicables a las operaciones generadas.

### 9.2 Cost per Rebalance
Cálculo de los costes asociados a los cambios de posiciones en cada periodo.

### 9.3 Gross Portfolio Returns
Cálculo de los retornos de las carteras antes de considerar los costes de transacción.

### 9.4 Net Portfolio Returns
Descuento de los costes de transacción para obtener los retornos netos de cada estrategia.

## 10. Portfolio Comparison
Comparamos las diferentes combinaciones de modelo, selección y metodología de weighting bajo unas condiciones de construcción homogéneas.

### 10.1 Ridge Portfolios
Construcción y almacenamiento de las carteras generadas a partir de las predicciones de Ridge utilizando las distintas metodologías de asignación de pesos.

### 10.2 XGBoost Portfolios
Construcción y almacenamiento de las carteras generadas a partir de las predicciones de XGBoost utilizando las mismas reglas.

### 10.3 Random Forest Portfolios
Construcción y almacenamiento de las carteras generadas a partir de las predicciones de Random Forest utilizando las mismas reglas.

### 10.4 Signal Comparison
Comparación de los rankings y de las selecciones producidas por los distintos modelos para analizar cómo las diferencias en las señales afectan a la composición de las carteras.

### 10.5 Weighting Comparison
Comparación de cómo las diferentes metodologías de asignación transforman una misma señal en carteras con diferentes niveles de concentración, riesgo y exposición.

### 10.6 Turnover & Cost Comparison
Comparación del turnover y de los costes de transacción generados por cada combinación de modelo y metodología de construcción.

## 11. Export Results
Guardamos los resultados necesarios para realizar posteriormente el análisis financiero completo de las estrategias en el Notebook 09.

### 11.1 Portfolio Weights
Exportación de los pesos de las carteras en cada fecha de rebalanceo.

### 11.2 Portfolio Returns
Exportación de los retornos brutos y netos de las estrategias.

### 11.3 Turnover
Exportación del turnover generado por cada cartera y periodo.

### 11.4 Transaction Costs
Exportación de los costes de transacción aplicados a cada estrategia.

### 11.5 Portfolio Metadata
Registro de los parámetros, reglas de selección, metodología de weighting, restricciones y supuestos utilizados para construir cada cartera.

## 12. Conclusions
Resumen de las principales características de las carteras construidas y de las diferencias observadas entre modelos, reglas de selección y metodologías de asignación de pesos.